<a href="https://colab.research.google.com/github/projectosmili/Age-Calculator-/blob/main/Hackathon_UniCesumar_%2B_Qlik_2026_(ForgeCloud).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏗️ Hackathon UniCesumar + Qlik 2026

## Análise da Distribuição dos Recursos da Reconstrução no Rio Grande do Sul

### Objetivo do Projeto

Este notebook tem como objetivo preparar e padronizar as bases de dados utilizadas no Hackathon Unicesumar 2026.

A partir dos dados de pagamentos públicos, impacto territorial, indicadores socioeconômicos e informações demográficas, serão construídas tabelas analíticas para posterior utilização no Qlik Cloud.

O foco da análise é responder às seguintes perguntas:

1. Onde os recursos foram aplicados?
2. Os recursos ficaram concentrados em determinados municípios?
3. Os municípios mais afetados receberam mais recursos?
4. Os municípios mais vulneráveis foram priorizados?
5. A distribuição dos recursos foi coerente com o impacto observado?

---

# 1. Configuração do Ambiente

## Objetivo

Configurar o ambiente de trabalho, conectar o Google Drive e definir os diretórios utilizados durante o processo de preparação dos dados.

As bibliotecas utilizadas serão:

- pandas
- numpy
- openpyxl
- unidecode

Os arquivos serão carregados diretamente da pasta do projeto no Google Drive.

---

# 1. Configuração do Ambiente
## 1.1 Importação das Bibliotecas

In [ ]:
!pip install unidecode -q

import pandas as pd
import numpy as np
import os
from unidecode import unidecode
from google.colab import drive

# Montar o Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Definição dos Diretórios do Projeto

Os arquivos serão organizados em duas camadas:

### data_raw

Contém os arquivos originais obtidos nas fontes de dados.

Os arquivos dessa pasta não serão sobrescritos ou modificados durante o tratamento.

### data_clean

Contém as tabelas tratadas, padronizadas e preparadas para carga no Qlik Cloud.

Essa separação permite:

- Preservar os dados originais
- Garantir rastreabilidade
- Reexecutar o processo de tratamento
- Evitar sobrescrita acidental das fontes
- Diferenciar os arquivos de entrada dos produtos analíticos
``

In [ ]:
# Pasta principal do projeto

pasta_projeto = (
    "/content/drive/MyDrive/"
    "Hackathon Unicesumar Qlik 2026"
)

# Pasta dos arquivos originais

pasta_raw = os.path.join(
    pasta_projeto,
    "data_raw"
)

# Pasta dos arquivos tratados

pasta_clean = os.path.join(
    pasta_projeto,
    "data_clean"
)

# Criar as pastas caso ainda não existam

os.makedirs(
    pasta_raw,
    exist_ok=True
)

os.makedirs(
    pasta_clean,
    exist_ok=True
)

print(
    "Pasta do projeto:",
    pasta_projeto
)

print(
    "Pasta dos dados originais:",
    pasta_raw
)

print(
    "Pasta dos dados tratados:",
    pasta_clean
)

Pasta do projeto: /content/drive/MyDrive/Hackathon Unicesumar Qlik 2026
Pasta dos dados originais: /content/drive/MyDrive/Hackathon Unicesumar Qlik 2026/data_raw
Pasta dos dados tratados: /content/drive/MyDrive/Hackathon Unicesumar Qlik 2026/data_clean


# 2. Inventário dos Arquivos

## Objetivo

Identificar os arquivos disponíveis na pasta do projeto e validar quais bases serão utilizadas na construção do modelo analítico.

As bases esperadas são:

### Pagamentos

Base financeira contendo os repasses realizados durante a reconstrução da crise climática.

Tabela destino:

FACT_PAGAMENTOS

---

### IBGE

Base de municípios do Rio Grande do Sul contendo informações geográficas e demográficas.

Tabela destino:

DIM_MUNICIPIO

---

### ADH (Atlas do Desenvolvimento Humano)

Base de indicadores de desenvolvimento humano, educação, renda, desigualdade e condições socioeconômicas disponibilizada pelo Atlas do Desenvolvimento Humano no Brasil.

Para este projeto, foi utilizado o recorte da Unidade da Federação do Rio Grande do Sul referente ao ano de 2024, permitindo contextualizar os indicadores sociais e econômicos do estado.

Tabela destino:

DIM_INDICADORES_SOCIAIS_RS

---

### Impacto Territorial

Base derivada dos estudos especiais e clusters do MUP-RS.

Tabela destino:

FACT_IMPACTO_MUNICIPIO

---

Nesta etapa serão identificados:

- Nome dos arquivos
- Formato dos arquivos
- Tamanho das bases
- Disponibilidade dos dados necessários para o projeto

## Inventário dos Arquivos por Camada

Nesta etapa serão listados separadamente:

- Arquivos originais disponíveis em `data_raw`
- Arquivos tratados disponíveis em `data_clean`

In [ ]:
# Listar os arquivos originais

arquivos_raw = sorted(
    os.listdir(
        pasta_raw
    )
)

print("ARQUIVOS EM DATA_RAW")
print("-" * 50)

for arquivo in arquivos_raw:
    print(arquivo)

ARQUIVOS EM DATA_RAW
--------------------------------------------------
Clusters.xlsx
DIM_CREDOR_CNPJ.csv
FACT_PAGAMENTOS_BIGQUERY.csv
IBGE_Municipios_Rio_Grande_do_Sul.xlsx
Pagamentos.xlsx
REPASSE_MUNICIPIO.csv
adh_radar_base_2012_2024.xlsx
base_de_dados_UDH.xlsx


In [ ]:
# Listar os arquivos tratados

arquivos_clean = sorted(
    os.listdir(
        pasta_clean
    )
)

print("\nARQUIVOS EM DATA_CLEAN")
print("-" * 50)

if arquivos_clean:

    for arquivo in arquivos_clean:
        print(arquivo)

else:

    print(
        "A pasta data_clean ainda está vazia."
    )


ARQUIVOS EM DATA_CLEAN
--------------------------------------------------
DIM_CREDOR_CNPJ.xlsx
DIM_INDICADORES_SOCIAIS_RS.xlsx
DIM_MUNICIPIO.xlsx
FACT_IMPACTO_MUNICIPIO.xlsx
FACT_PAGAMENTOS.xlsx
FACT_REPASSE_MUNICIPIO.xlsx
PAGAMENTOS.xlsx


# 3. Carregamento das Bases de Dados

## Objetivo

Carregar as bases que serão utilizadas na construção do modelo analítico do projeto.

As tabelas principais do modelo serão:

### DIM_MUNICIPIO

Fonte:

IBGE_Municipios_Rio_Grande_do_Sul.xlsx

Objetivo:

Concentrar informações demográficas e territoriais dos municípios do Rio Grande do Sul.

---

### FACT_PAGAMENTOS

Fonte:

Pagamentos.xlsx

Objetivo:

Concentrar os repasses financeiros relacionados à reconstrução da crise climática.

---

### FACT_IMPACTO_MUNICIPIO

Fonte:

Clusters.xlsx

Objetivo:

Concentrar os indicadores de impacto territorial e população afetada.

---

### DIM_INDICADORES_SOCIAIS_RS

Fonte:

adh_radar_base_2012_2024.xlsx

Objetivo:

Concentrar indicadores socioeconômicos, educacionais e de desenvolvimento humano do Rio Grande do Sul, fornecendo contexto social para análise dos impactos climáticos e dos investimentos realizados.

Indicadores principais:

- IDHM
- IDHM_E
- IDHM_R
- IDHM_L
- RDPC
- GINI
- THEIL
- ESPVIDA
- ANOSEST
- T_ANALF15M
- T_SUPER25M
- PIND
- PMPOB
- PPOB

---

Nesta etapa serão avaliados:

- Quantidade de registros
- Quantidade de colunas
- Tipos de dados
- Qualidade dos campos
- Chaves de relacionamento
- Cobertura geográfica dos dados

In [ ]:
# Arquivos principais

arquivo_pagamentos = os.path.join(
    pasta_raw,
    'Pagamentos.xlsx'
)

arquivo_ibge = os.path.join(
    pasta_raw,
    'IBGE_Municipios_Rio_Grande_do_Sul.xlsx'
)

arquivo_impacto = os.path.join(
    pasta_raw,
    'Clusters.xlsx'
)

# Carregar Pagamentos

In [ ]:
pagamentos = pd.read_excel(arquivo_pagamentos)

print("PAGAMENTOS")
print("-"*50)
print(f"Linhas : {pagamentos.shape[0]}")
print(f"Colunas: {pagamentos.shape[1]}")

pagamentos.head()

PAGAMENTOS
--------------------------------------------------
Linhas : 38210
Colunas: 22


,Data,Fase Gasto,Empenho,CPF / CNPJ,Credor,Órgão,Valor,Processo,Poder,Unidade Orçamentária,...,Função,Subfunção,Ação Programática,Iniciativa,Projeto,Subprojeto,Grupo de Despesa,Modalidade,Elemento,Rubrica
0,2024-12-19,Pago,24007402766,46191353000117,Portos Rs,Secretaria de Logistica e Transportes,731389734.0,24930100024475,Poder Executivo,Gabinete e Orgaos Centrais,...,Transporte,Administracao Financeira,Encargos Especiais - Selt,Capitalizacao de Empresas Estatais,Aumento de Capital Em Vinculada-portos Rs,Aumento de Capital - Portos Rs,Inversoes Financeiras,Aplicacoes Diretas,Constituicao Ou Aumento de Capital de Empresas,Participacao Em Constituicao Ou Aumento de Cap...
1,2026-01-13,Pago,25008797146,56109224000190,Agencia de Desenvolvimento do Rio Grande do Su,Secretaria de Desenvolvimento Economico,200000000.0,25230100004041,Poder Executivo,Gabinete e Orgaos Centrais,...,Industria,Promocao Industrial,Invest Rs,Atracao de Investimentos,Implantacao e Desenvolvimento do Servico Socia...,Progantur-dec 58342/2025,Outras Despesas Correntes,Transferencias a Instituicoes Privadas Sem Fin...,Contrato de Gestao,Contrato de Gestao Invest Rs
2,2025-05-28,Pago,25002232561,99999999999962,Fbr Aviation Inc,SSP - Brigada Militar,170789344.0,25120300137211,Poder Executivo,Brigada Militar,...,Seguranca Publica,Policiamento,Rs Mais Seguro,Fortalecimento da Capacidade de Resposta Ao Ci...,Qualificacao das Instalacoes e Servicos da B...,Aquiscao de Aeronaves - Enfrentamento Enchente...,Investimentos,Aplicacoes Diretas,Equipamentos e Material Permanente,Aeronaves E/ou Equipamentos Para Aeronaves
3,2025-09-19,Pago,25005103752,00360305000104,Caixa Economica Federal,Secretaria de Habitacao e Regularizacao Fundiaria,150000000.0,24170000005604,Poder Executivo,Fundo Estadual de Habitacao de Interesse Social,...,Habitacao,Habitacao Urbana,Acoes Habitacionais e Regularizacao Fundiaria,Promocao de Acoes Habitacionais,Porta de Entrada,Porta de Entrada Cidadao,Outras Despesas Correntes,Aplicacoes Diretas,Outros Auxilios Financeiros a Pessoas Fisicas,Subsidios Programa Porta de Entrada
4,2025-12-10,Pago,25008248126,92702067000196,Banco do Estado do Rio Grande do Sul S/a,"Secretaria da Agricultura, Pecuaria, Producao ...",150000000.0,25147100002705,Poder Executivo,Gabinete e Orgaos Centrais .,...,Agricultura,Irrigacao,Supera Estiagem,Apoio a Infraestrutura Hidrica Rural,Subvencao a Produtores Rurais,Subvencao Renegociacao Dividas de Credito Rural,Outras Despesas Correntes,Aplicacoes Diretas,Subvencoes Economicas,Outras Equalizacoes de Juros


# Carregar IBGE

In [ ]:
ibge = pd.read_excel(arquivo_ibge)

print("IBGE")
print("-"*50)
print(f"Linhas : {ibge.shape[0]}")
print(f"Colunas: {ibge.shape[1]}")

ibge.head()


IBGE
--------------------------------------------------
Linhas : 512
Colunas: 14


,Município [-],Código [-],Gentílico [-],Prefeito [2025],Área Territorial - km² [2025],População no último censo - pessoas [2022],Densidade demográfica - hab/km² [2022],População estimada - pessoas [2025],Escolarização <span>6 a 14 anos</span> - % [2022],IDHM <span>Índice de desenvolvimento humano municipal</span> [2010],Mortalidade infantil - óbitos por mil nascidos vivos [2025],Total de receitas brutas realizadas - R$ [2025],Total de despesas brutas empenhadas - R$ [2025],PIB per capita - R$ [2023]
0,Aceguá,4300034.0,aceguaense,MARCUS VINÍCIUS GODOY DE AGUIAR,1551.339,4170.0,2.69,4251.0,98.80,0.687,34.48,6.856418e+07,6.167138e+07,91688.37
1,Água Santa,4300059.0,água-santense,JULIANO FAVRETTO,291.526,3912.0,13.42,4003.0,99.18,0.75,27.78,5.473052e+07,4.185100e+07,107039.28
2,Agudo,4300109.0,agudense,LUIS HENRIQUE KITTEL,534.624,16041.0,30.00,16341.0,100.00,0.694,19.35,1.315546e+08,1.286572e+08,48634.87
3,Ajuricaba,4300208.0,ajuricabense,PAULO CLAUDIO DOLOVITSCH,322.674,6720.0,20.83,6843.0,100.00,0.753,16.39,6.216223e+07,5.420858e+07,57072.14
4,Alecrim,4300307.0,alecrinense,NEUSA LEDUR KUHN,316.394,6123.0,19.35,6219.0,99.26,0.672,-,4.781322e+07,4.560098e+07,25105.18


# Carregar Impacto

In [ ]:
impacto = pd.read_excel(arquivo_impacto)

print("IMPACTO")
print("-"*50)
print(f"Linhas : {impacto.shape[0]}")
print(f"Colunas: {impacto.shape[1]}")

impacto.head()

IMPACTO
--------------------------------------------------
Linhas : 155
Colunas: 9


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Ordem,Cluster,População,Área (ha),Densidade Populacional (pop/ha),% Pop. do Município,Nº de CNPJs,Recorrência,Escore
0,1,Muçum_1,2497.284293,97.656602,25.572099,0.542770,402.0,Sim,7.375771
1,2,Relvado_1,324.169809,16.217296,19.989141,0.180495,93.0,Não,7
2,3,Marques de Souza_1,916.307086,42.514969,21.552576,0.230866,171.0,Não,6.303384
3,4,Cruzeiro do Sul_1,362.179716,9.572855,37.834034,0.031222,19.0,Sim,6.074208
4,5,Cristal do Sul_1,413.489856,18.644226,22.177904,0.153600,123.0,Não,6


# Inspeção das Colunas

In [ ]:
print("\nCOLUNAS PAGAMENTOS\n")
print(pagamentos.columns.tolist())

print("\nCOLUNAS IBGE\n")
print(ibge.columns.tolist())

print("\nCOLUNAS IMPACTO\n")
print(impacto.columns.tolist())


COLUNAS PAGAMENTOS

['Data', 'Fase Gasto', 'Empenho', 'CPF / CNPJ', 'Credor', 'Órgão', 'Valor', 'Processo', 'Poder', 'Unidade Orçamentária', 'Recurso', 'Fonte Recurso', 'Função', 'Subfunção', 'Ação Programática', 'Iniciativa', 'Projeto', 'Subprojeto', 'Grupo de Despesa', 'Modalidade', 'Elemento', 'Rubrica']

COLUNAS IBGE

['Município [-]', 'Código [-]', 'Gentílico [-]', 'Prefeito [2025]', 'Área Territorial - km² [2025]', 'População no último censo - pessoas [2022]', 'Densidade demográfica - hab/km² [2022]', 'População estimada - pessoas [2025]', 'Escolarização <span>6 a 14 anos</span> - % [2022]', 'IDHM <span>Índice de desenvolvimento humano municipal</span> [2010]', 'Mortalidade infantil - óbitos por mil nascidos vivos [2025]', 'Total de receitas brutas realizadas - R$ [2025]', 'Total de despesas brutas empenhadas - R$ [2025]', 'PIB per capita - R$ [2023]']

COLUNAS IMPACTO

['Ordem', 'Cluster', 'População', 'Área (ha)', 'Densidade Populacional (pop/ha)', '% Pop. do Município', 'Nº d

# Inspeção de Tipos

In [ ]:
print("\nTIPOS PAGAMENTOS")
pagamentos.info()

print("\nTIPOS IBGE")
ibge.info()

print("\nTIPOS IMPACTO")
impacto.info()


TIPOS PAGAMENTOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38210 entries, 0 to 38209
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Data                  38210 non-null  datetime64[ns]
 1   Fase Gasto            38210 non-null  object        
 2   Empenho               38210 non-null  int64         
 3   CPF / CNPJ            38210 non-null  object        
 4   Credor                38210 non-null  object        
 5   Órgão                 38210 non-null  object        
 6   Valor                 38210 non-null  float64       
 7   Processo              38210 non-null  int64         
 8   Poder                 38210 non-null  object        
 9   Unidade Orçamentária  38210 non-null  object        
 10  Recurso               38210 non-null  object        
 11  Fonte Recurso         38210 non-null  object        
 12  Função                38210 non-null  object        
 13

# 4. Construção da DIM_MUNICIPIO

## Objetivo

Construir a dimensão principal de municípios a partir da base oficial do IBGE.

Esta dimensão será utilizada como tabela mestre do modelo analítico e servirá como referência para integração dos dados financeiros, demográficos e de impacto territorial.

A tabela deverá possuir apenas um registro por município.

## Tabela de Origem

IBGE_Municipios_Rio_Grande_do_Sul.xlsx

## Campos Selecionados

- Código IBGE
- Município
- Área Territorial
- População (Censo 2022)
- População Estimada (2025)
- Densidade Demográfica
- IDHM
- Receitas Municipais
- Despesas Municipais
- PIB per Capita

## Resultado Esperado

DIM_MUNICIPIO

Uma tabela contendo um único registro para cada município do Rio Grande do Sul.



# Seleção dos Campos

In [ ]:
dim_municipio = ibge[[
    'Código [-]',
    'Município [-]',
    'Área Territorial - km² [2025]',
    'População no último censo - pessoas [2022]',
    'População estimada - pessoas [2025]',
    'Densidade demográfica - hab/km² [2022]',
    'IDHM <span>Índice de desenvolvimento humano municipal</span> [2010]',
    'Total de receitas brutas realizadas - R$ [2025]',
    'Total de despesas brutas empenhadas - R$ [2025]',
    'PIB per capita - R$ [2023]'
]].copy()

# Renomeação das Colunas

In [ ]:
dim_municipio.columns = [
    'Cod_IBGE',
    'Municipio',
    'Area_km2',
    'Populacao_Censo_2022',
    'Populacao_Estimada_2025',
    'Densidade',
    'IDHM',
    'Receita_Municipal',
    'Despesa_Municipal',
    'PIB_Per_Capita'
]

# Limpeza dos Registros Vazios

In [ ]:
dim_municipio = dim_municipio.dropna(
    subset=['Cod_IBGE']
)

# Conversão do Código IBGE

In [ ]:
dim_municipio['Cod_IBGE'] = (
    dim_municipio['Cod_IBGE']
    .astype(int)
)

# Remoção de Duplicidades

In [ ]:
dim_municipio = dim_municipio.drop_duplicates(
    subset=['Cod_IBGE']
)

# Verificação da Estrutura

In [ ]:
print("Quantidade de municípios:")
print(dim_municipio.shape[0])

print("\nQuantidade de colunas:")
print(dim_municipio.shape[1])

dim_municipio.head()

Quantidade de municípios:
497

Quantidade de colunas:
10


,Cod_IBGE,Municipio,Area_km2,Populacao_Censo_2022,Populacao_Estimada_2025,Densidade,IDHM,Receita_Municipal,Despesa_Municipal,PIB_Per_Capita
0,4300034,Aceguá,1551.339,4170.0,4251.0,2.69,0.687,6.856418e+07,6.167138e+07,91688.37
1,4300059,Água Santa,291.526,3912.0,4003.0,13.42,0.75,5.473052e+07,4.185100e+07,107039.28
2,4300109,Agudo,534.624,16041.0,16341.0,30.00,0.694,1.315546e+08,1.286572e+08,48634.87
3,4300208,Ajuricaba,322.674,6720.0,6843.0,20.83,0.753,6.216223e+07,5.420858e+07,57072.14
4,4300307,Alecrim,316.394,6123.0,6219.0,19.35,0.672,4.781322e+07,4.560098e+07,25105.18


# Verificação de Nulos

In [ ]:
dim_municipio.isnull().sum()

,0
Cod_IBGE,0
Municipio,0
Area_km2,0
Populacao_Censo_2022,0
Populacao_Estimada_2025,0
Densidade,0
IDHM,0
Receita_Municipal,0
Despesa_Municipal,0
PIB_Per_Capita,0


# Estatísticas Básicas

In [ ]:
dim_municipio.describe(
    include='all'
)

,Cod_IBGE,Municipio,Area_km2,Populacao_Censo_2022,Populacao_Estimada_2025,Densidade,IDHM,Receita_Municipal,Despesa_Municipal,PIB_Per_Capita
count,4.970000e+02,497,497.000000,4.970000e+02,4.970000e+02,497.000000,497.000,4.970000e+02,4.970000e+02,497.000000
unique,NaN,497,NaN,NaN,NaN,NaN,158.000,NaN,NaN,NaN
top,NaN,Xangri-lá,NaN,NaN,NaN,NaN,0.739,NaN,NaN,NaN
freq,NaN,1,NaN,NaN,NaN,NaN,11.000,NaN,NaN,NaN
mean,4.311877e+06,NaN,540.489789,2.189731e+04,2.260214e+04,91.591026,NaN,1.761560e+08,1.647070e+08,54192.573099
std,6.683443e+03,NaN,908.131055,7.374416e+04,7.668319e+04,324.446420,NaN,5.765265e+08,6.103517e+08,29292.255947
min,4.300034e+06,NaN,27.676000,1.135000e+03,1.156000e+03,1.500000,NaN,2.231199e+07,2.214470e+07,19953.250000
25%,4.306106e+06,NaN,125.334000,2.877000e+03,2.937000e+03,12.970000,NaN,4.472716e+07,3.730050e+07,36907.080000
50%,4.312104e+06,NaN,236.653000,5.378000e+03,5.490000e+03,21.790000,NaN,5.846134e+07,4.997801e+07,47185.140000
75%,4.317400e+06,NaN,504.114000,1.495500e+04,1.541300e+04,43.370000,NaN,1.246716e+08,1.065387e+08,61786.880000


# Exportação

In [ ]:
arquivo_saida = os.path.join(
    pasta_clean,
    'DIM_MUNICIPIO.xlsx'
)

dim_municipio.to_excel(
    arquivo_saida,
    index=False
)

print("✅ DIM_MUNICIPIO.xlsx criado com sucesso.")

✅ DIM_MUNICIPIO.xlsx criado com sucesso.


# Validação Final

In [ ]:
print(
    f"Municípios únicos: {dim_municipio['Municipio'].nunique()}"
)

print(
    f"Códigos IBGE únicos: {dim_municipio['Cod_IBGE'].nunique()}"
)

Municípios únicos: 497
Códigos IBGE únicos: 497


# 5. Construção da FACT_IMPACTO_MUNICIPIO

## Objetivo

Construir a tabela de impacto territorial consolidando os clusters afetados por município.

A base original apresenta múltiplos clusters para um mesmo município, representando áreas específicas afetadas pelos eventos climáticos.

Exemplos:

- Muçum_1
- Cruzeiro do Sul_1
- Cruzeiro do Sul_2
- Cruzeiro do Sul_3

O objetivo desta etapa é consolidar essas ocorrências em um único registro municipal, criando indicadores agregados que possam ser comparados posteriormente com os repasses financeiros e indicadores socioeconômicos.

---

## Tabela de Origem

Clusters.xlsx

---

## Tratamentos Aplicados

### Extração do Município

Remoção do identificador do cluster para obter apenas o nome do município.

Exemplo:

```text
Cruzeiro do Sul_1 → Cruzeiro do Sul
Cruzeiro do Sul_2 → Cruzeiro do Sul
Cruzeiro do Sul_3 → Cruzeiro do Sul
```

---

### Consolidação Municipal

Os registros serão agrupados por município.

Para cada indicador será aplicada uma regra de agregação apropriada:

- População afetada → Soma
- Área afetada → Soma
- Densidade afetada → Média
- Percentual da população afetada → Máximo
- CNPJs afetados → Soma
- Recorrência → Máximo
- Escore → Média

---

## Resultado Esperado

FACT_IMPACTO_MUNICIPIO

Campos finais:

- Municipio
- Pop_Afetada
- Area_Afetada_Ha
- Densidade_Afetada
- Perc_Pop_Afetada_Max
- CNPJs_Afetados
- Recorrencia
- Escore_Medio

Esta tabela será utilizada para avaliar o impacto territorial da crise climática e posteriormente

# Extração do Município

In [ ]:
impacto['Municipio'] = (
    impacto['Cluster']
    .str.replace(r'_[0-9]+$', '', regex=True)
)

# Verificação da Extração

In [ ]:
impacto[['Cluster', 'Municipio']].head(20)

,Cluster,Municipio
0,Muçum_1,Muçum
1,Relvado_1,Relvado
2,Marques de Souza_1,Marques de Souza
3,Cruzeiro do Sul_1,Cruzeiro do Sul
4,Cristal do Sul_1,Cristal do Sul
5,Dois Lajeados_1,Dois Lajeados
6,Entre Rios do Sul_1,Entre Rios do Sul
7,Cruzeiro do Sul_2,Cruzeiro do Sul
8,Nova Bassano_1,Nova Bassano
9,Encantado_1,Encantado


# Remoção de Registros Vazios

In [ ]:
impacto_limpo = impacto.dropna(
    subset=['Municipio']
).copy()

# Tratamento do Campo Escore

In [ ]:
impacto_limpo['Escore'] = (
    impacto_limpo['Escore']
    .astype(str)
    .str.replace(',', '.', regex=False)
)

impacto_limpo['Escore'] = pd.to_numeric(
    impacto_limpo['Escore'],
    errors='coerce'
)

# Consolidação por Município

In [ ]:
fact_impacto = (
    impacto_limpo
    .groupby('Municipio')
    .agg({
        'População': 'sum',
        'Área (ha)': 'sum',
        'Densidade Populacional (pop/ha)': 'mean',
        '% Pop. do Município': 'max',
        'Nº de CNPJs': 'sum',
        'Recorrência': 'max',
        'Escore': 'mean'
    })
    .reset_index()
)

# Renomeação das Colunas

In [ ]:
fact_impacto.columns = [
    'Municipio',
    'Pop_Afetada',
    'Area_Afetada_Ha',
    'Densidade_Afetada',
    'Perc_Pop_Afetada_Max',
    'CNPJs_Afetados',
    'Recorrencia',
    'Escore_Medio'
]

# Verificação da Estrutura

In [ ]:
print("Quantidade de municípios:")

print(fact_impacto.shape[0])

print("\nQuantidade de colunas:")

print(fact_impacto.shape[1])

fact_impacto.head()

Quantidade de municípios:
68

Quantidade de colunas:
8


,Municipio,Pop_Afetada,Area_Afetada_Ha,Densidade_Afetada,Perc_Pop_Afetada_Max,CNPJs_Afetados,Recorrencia,Escore_Medio
0,Alvorada,28827.073334,587.133627,70.822313,0.151742,2319.0,Não,2.939899
1,Arambaré,1780.914216,145.772707,12.184334,0.175792,161.0,Não,4.571723
2,Arroio Grande,362.475988,16.636933,21.787428,0.020644,11.0,Não,3.928917
3,Arroio do Meio,6616.814504,223.225613,31.434256,0.144617,1067.0,Sim,4.436807
4,Arroio do Tigre,1000.624633,22.906113,43.683737,0.082984,37.0,Não,4.278957


# Verificação de Valores Nulos

In [ ]:
fact_impacto.isnull().sum()

,0
Municipio,0
Pop_Afetada,0
Area_Afetada_Ha,0
Densidade_Afetada,0
Perc_Pop_Afetada_Max,0
CNPJs_Afetados,0
Recorrencia,0
Escore_Medio,0


# Estatísticas Básicas

In [ ]:
fact_impacto.describe(
    include='all'
)

,Municipio,Pop_Afetada,Area_Afetada_Ha,Densidade_Afetada,Perc_Pop_Afetada_Max,CNPJs_Afetados,Recorrencia,Escore_Medio
count,68,68.000000,68.000000,68.000000,68.000000,68.000000,68,68.000000
unique,68,NaN,NaN,NaN,NaN,NaN,2,NaN
top,Alvorada,NaN,NaN,NaN,NaN,NaN,Não,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,47,NaN
mean,NaN,11841.572307,289.859905,41.551993,0.133639,1414.617647,NaN,4.202616
std,NaN,27521.358376,627.642237,17.738260,0.140300,3632.722777,NaN,1.005920
min,NaN,324.169809,8.458469,11.413196,0.004196,11.000000,NaN,2.548036
25%,NaN,1176.579150,28.811541,27.838027,0.028874,139.000000,NaN,3.536319
50%,NaN,3146.496081,88.309616,39.315179,0.083245,257.500000,NaN,3.977731
75%,NaN,7510.681296,223.861792,52.346772,0.173363,991.250000,NaN,4.567834


# Validação Final

In [ ]:
print(
    f"Municípios únicos: {fact_impacto['Municipio'].nunique()}"
)
print(
    f"Registros totais: {len(fact_impacto)}"
)
fact_impacto.sort_values(
    'Pop_Afetada',
    ascending=False
).head(10)


Municípios únicos: 68
Registros totais: 68


,Municipio,Pop_Afetada,Area_Afetada_Ha,Densidade_Afetada,Perc_Pop_Afetada_Max,CNPJs_Afetados,Recorrencia,Escore_Medio
11,Canoas,158553.721200,4298.973155,36.881766,0.456064,19987.0,Não,3.285729
57,São Leopoldo,109404.746332,2173.335820,70.258925,0.481805,11662.0,Sim,3.742099
39,Porto Alegre,109255.337110,1390.127427,86.334162,0.022496,19091.0,Não,2.548036
42,Rio Grande,67161.439355,1488.330034,43.910371,0.321322,7382.0,Não,2.555293
38,Pelotas,48789.321208,1079.662347,51.608111,0.099695,4333.0,Não,3.315413
18,Eldorado do Sul,30190.860968,781.405355,76.191670,0.576504,2378.0,Não,4.317247
26,Guaíba,30128.983217,587.741481,47.358954,0.229137,1788.0,Não,3.514465
0,Alvorada,28827.073334,587.133627,70.822313,0.151742,2319.0,Não,2.939899
22,Esteio,19400.542523,297.602018,81.417600,0.169798,1534.0,Sim,4.831698
27,Igrejinha,15506.030620,662.631957,23.400668,0.472630,2871.0,Sim,4.629868


# 6. Integração e Padronização da FACT_IMPACTO_MUNICIPIO

## Objetivo

Adicionar o Código IBGE à FACT_IMPACTO_MUNICIPIO utilizando a DIM_MUNICIPIO como tabela de referência.

A utilização do Código IBGE garante:

- padronização dos relacionamentos
- integração com outras bases
- redução de erros de nomenclatura
- maior qualidade analítica

## Estratégia

A correspondência será realizada utilizando o nome do município.

Após a integração, cada registro de impacto deverá possuir um Código IBGE válido.

## Resultado Esperado

FACT_IMPACTO_MUNICIPIO

Campos:

- Cod_IBGE
- Municipio
- Pop_Afetada
- Area_Afetada_Ha
- Densidade_Afetada
- Perc_Pop_Afetada_Max
- CNPJs_Afetados
- Recorrencia
- Escore_Medio

# Merge

In [ ]:
fact_impacto = fact_impacto.merge(
    dim_municipio[
        ['Cod_IBGE', 'Municipio']
    ],
    on='Municipio',
    how='left'
)


# Reordenar colunas

In [ ]:
fact_impacto = fact_impacto[
    [
        'Cod_IBGE',
        'Municipio',
        'Pop_Afetada',
        'Area_Afetada_Ha',
        'Densidade_Afetada',
        'Perc_Pop_Afetada_Max',
        'CNPJs_Afetados',
        'Recorrencia',
        'Escore_Medio'
    ]
]

# Validação

In [ ]:
fact_impacto['Cod_IBGE'].isnull().sum()

np.int64(1)

# Se aparecerem nulos

In [ ]:
fact_impacto[
    fact_impacto['Cod_IBGE'].isnull()
][['Municipio']]

,Municipio
41,Restinga Seca


# Estatísticas

In [ ]:
print(
    f"Municípios com Código IBGE: "
    f"{fact_impacto['Cod_IBGE'].notnull().sum()}"
)

print(
    f"Municípios sem Código IBGE: "
    f"{fact_impacto['Cod_IBGE'].isnull().sum()}"
)

Municípios com Código IBGE: 67
Municípios sem Código IBGE: 1


# Vamos fazer o diagnóstico

In [ ]:
dim_municipio[
    dim_municipio['Municipio']
    .str.contains('Restinga', case=False, na=False)
]

,Cod_IBGE,Municipio,Area_km2,Populacao_Censo_2022,Populacao_Estimada_2025,Densidade,IDHM,Receita_Municipal,Despesa_Municipal,PIB_Per_Capita
341,4315503,Restinga Sêca,968.62,14939.0,15205.0,15.42,0.683,124103062.0,1.040674e+08,43057.1


# Criar chaves padronizadas

In [ ]:
def normalizar_municipio(texto):

    if pd.isna(texto):
        return None

    return (
        unidecode(str(texto))
        .upper()
        .strip()
    )

dim_municipio['Municipio_Key'] = (
    dim_municipio['Municipio']
    .apply(normalizar_municipio)
)

fact_impacto['Municipio_Key'] = (
    fact_impacto['Municipio']
    .apply(normalizar_municipio)
)

# Fazer a Merge novamente

In [ ]:
fact_impacto = fact_impacto.drop(
    columns=['Cod_IBGE'],
    errors='ignore'
)


In [ ]:
fact_impacto = fact_impacto.merge(
    dim_municipio[
        [
            'Cod_IBGE',
            'Municipio_Key'
        ]
    ],
    on='Municipio_Key',
    how='left'
)

# Reordenar colunas

In [ ]:
fact_impacto = fact_impacto[
    [
        'Cod_IBGE',
        'Municipio',
        'Pop_Afetada',
        'Area_Afetada_Ha',
        'Densidade_Afetada',
        'Perc_Pop_Afetada_Max',
        'CNPJs_Afetados',
        'Recorrencia',
        'Escore_Medio'
    ]
]

# Validar

In [ ]:
print(
    f"Municípios sem Código IBGE: "
    f"{fact_impacto['Cod_IBGE'].isnull().sum()}"
)

Municípios sem Código IBGE: 0


# Renomear o município da tabela de impacto

In [ ]:
# Renomear o município da tabela de impacto
# para evitar associação dupla no Painel

fact_impacto = fact_impacto.rename(
    columns={
        "Municipio":
        "Municipio_Impactado"
    }
)

# Exportação

In [ ]:
arquivo_saida = os.path.join(
    pasta_clean,
    'FACT_IMPACTO_MUNICIPIO.xlsx'
)

fact_impacto.to_excel(
    arquivo_saida,
    index=False
)

print(
    "✅ FACT_IMPACTO_MUNICIPIO.xlsx criado com sucesso."
)

✅ FACT_IMPACTO_MUNICIPIO.xlsx criado com sucesso.


# 7. Análise da FACT_PAGAMENTOS

## Objetivo

Compreender a estrutura da base de pagamentos e identificar possíveis mecanismos de relacionamento com os municípios do Rio Grande do Sul.

Diferentemente das bases anteriores, a tabela de pagamentos não possui uma coluna explícita de município.

Portanto, esta etapa tem como objetivo investigar os campos disponíveis e identificar possíveis chaves de municipalização dos recursos.

---

## Tabela de Origem

Pagamentos.xlsx

---

## Contexto

Durante a construção do Painel de Transparência da Crise Climática, a Secretaria da Fazenda do Rio Grande do Sul informou que parte da municipalização das despesas foi realizada utilizando diferentes informações administrativas como:

- Projeto
- Subprojeto
- Credor
- Órgão
- Informações complementares dos processos

Nem todas as despesas possuem um município associado de forma direta.

Existem recursos destinados a:

- Obras regionais
- Estradas
- Defesa Civil
- Equipamentos públicos
- Programas estaduais

Portanto, a identificação do município pode exigir tratamentos adicionais.

---

## Estrutura da Base

Principais campos disponíveis:

- Data
- Fase Gasto
- Empenho
- CPF / CNPJ
- Credor
- Órgão
- Valor
- Processo
- Poder
- Unidade Orçamentária
- Recurso
- Fonte Recurso
- Função
- Subfunção
- Ação Programática
- Iniciativa
- Projeto
- Subprojeto
- Grupo de Despesa
- Modalidade
- Elemento
- Rubrica

---

## Perguntas de Investigação

Nesta etapa serão respondidas as seguintes perguntas:

### Os municípios aparecem nos Projetos?

Verificar se existem projetos associados diretamente a municípios.

---

### Os municípios aparecem nos Subprojetos?

Verificar se os subprojetos possuem referências geográficas.

---

### Quais órgãos executaram mais recursos?

Identificar os responsáveis pelos maiores volumes de gastos.

---

### Quais credores receberam mais recursos?

Identificar os principais beneficiários dos repasses.

---

### Existe alguma chave de municipalização?

Investigar possíveis campos que permitam relacionar os pagamentos aos municípios afetados.

---

## Resultado Esperado

Definir a estratégia de integração entre os pagamentos e os municípios.

A conclusão desta etapa orientará a construção final da FACT_PAGAMENTOS e dos relacionamentos do modelo analítico.

# Principais Projetos

In [ ]:
pagamentos['Projeto'].value_counts().head(30)

,count
Projeto,
Conservacao de Rodovias,7176
Alimentacao Escolar Qualificada,6858
"Ampliacao, Construcao, Recuperacao e Reforma - Educacao Basica",2932
"Qualificacao dos Espacos Escolares - Equipamentos, Mobiliario e Material Pedagog",1698
Fiscalizacao de Transito Em Rodovias (bprv),1654
Manutencao dos Servicos de Bombeiros,1559
Manutencao dos Servicos de Policia Ostensiva,1379
Cofinanciamento de Servicos Socioassistenciais,1059
Atuacao da Defesa Civil Estadual,911


# Principais Subprojetos

In [ ]:
pagamentos['Subprojeto'].value_counts().head(50)

,count
Subprojeto,
Alimentacao Escolar Na Educacao Basica,6858
Manutencao da Estrutura Operacional,6477
Autonomia Financeira – Infraestrutura Eventos Climaticos,2894
Pessoal e Manutencao do Crbm,1651
"Autonomia Financeira – Mat, Equip e Mobiliario - Eventos Climaticos",1624
Manutencao dos Servicos de Bombeiros - Enfrentamento Enchentes 2024,1554
Manutencao dos Servicos de Policia Ostensiva,1379
Apoio Administrativo,811
Cofinanciamento Unificado - Piso Gaucho - Enfrentamento Enchentes 2024,732


# Principais Órgãos

In [ ]:
pagamentos['Órgão'].value_counts().head(30)

,count
Órgão,
Secretaria da Educacao,11529
Departamento Autonomo de Estradas de Rodagem,10261
Secretaria da Saude,2068
Secretaria de Desenvolvimento Urbano e Metropolitano,1726
SSP - Corpo de Bombeiros Militar,1635
Secretaria de Desenvolvimento Social,1486
SSP - Brigada Militar,1475
Governo do Estado,1100
Secretaria de Obras Publicas,952


# Principais Credores

In [ ]:
pagamentos['Credor'].value_counts().head(30)

,count
Credor,
Consorcio Desassoreamento Rs,534
Mak Servicos e Pavimentacoes Ltda,482
Agr Engenharia e Empreend Ltda,200
Rgs Engenharia S.a. - Em Recuperacao Judicial,187
Encopav Engenharia Ltda,186
Dalfovo Constr Ltda,118
Cmc Modulos Construtivos Ltda,116
Caixa Economica Federal,116
Tracado Const e Servs Ltda,107


# Maiores Valores por Projeto

In [ ]:
pagamentos.groupby(
    'Projeto',
    as_index=False
)['Valor'].sum().sort_values(
    'Valor',
    ascending=False
).head(30)

,Projeto,Valor
227,Restauracao e Manutencao de Malha Rodoviaria -...,7.447154e+08
73,Aumento de Capital Em Vinculada-portos Rs,7.313897e+08
86,Conservacao de Rodovias,4.592290e+08
205,Qualificacao das Instalacoes e Servicos da B...,3.794194e+08
195,Producao de Acoes Habitacionais,3.587856e+08
69,Atuacao da Defesa Civil Estadual,3.212755e+08
75,Auxilio Emergencial Rs - Sedes,2.605221e+08
17,Aperfeicoamento do Planejamento e Mobilidade U...,2.591929e+08
132,Gestao das Demandas do Plano Rio Grande,2.548625e+08
97,Correcao do Solo de Propriedades da Agricultur...,2.236212e+08


# Maiores Valores por Subprojeto

In [ ]:
pagamentos.groupby(
    'Subprojeto',
    as_index=False
)['Valor'].sum().sort_values(
    'Valor',
    ascending=False
).head(30)

,Subprojeto,Valor
88,Aumento de Capital - Portos Rs,7.313897e+08
122,Conservacao da Malha Pavimentada,4.137475e+08
20,Acoes Emergenciais - Fundo a Fundo,2.929911e+08
97,Auxilio a Familias Atingidas Por Eventos Clima...,2.520361e+08
68,Aquiscao de Aeronaves - Enfrentamento Enchente...,2.159281e+08
362,Porta de Entrada Cidadao,2.021252e+08
369,Progantur-dec 58342/2025,2.000000e+08
180,Ers-332 Ers-129 Km 0 Ao Km 92+010,1.694324e+08
402,Rede Hospitalar - Enfrentamento Enchentes 2024,1.670056e+08
139,Desassorear Rs,1.639404e+08


# 8. Construção da FACT_PAGAMENTOS

## Objetivo

Preparar a base financeira oficial para utilização no modelo analítico.

Esta tabela contém os registros de pagamentos, repasses e execuções financeiras relacionadas à reconstrução da crise climática.

Diferentemente das demais tabelas do modelo, esta base não possui relacionamento direto com municípios.

Seu propósito será apoiar análises de:

- volume financeiro
- órgãos executores
- programas financiados
- evolução temporal dos gastos
- tipos de despesa

---

## Tabela de Origem

Pagamentos.xlsx

---

## Resultado Esperado

FACT_PAGAMENTOS

Campos:

- Data
- Fase_Gasto
- Empenho
- Credor
- Orgao
- Valor
- Processo
- Poder
- Unidade_Orcamentaria
- Recurso
- Fonte_Recurso
- Funcao
- Subfuncao
- Acao_Programatica
- Iniciativa
- Projeto
- Subprojeto
- Grupo_Despesa
- Modalidade
- Elemento
- Rubrica

# Copiar pagamentos

In [ ]:
fact_pagamentos = pagamentos.copy()

# Renomear colunas

In [ ]:
fact_pagamentos.columns = [
    'Data',
    'Fase_Gasto',
    'Empenho',
    'CPF_CNPJ',
    'Credor',
    'Orgao',
    'Valor',
    'Processo',
    'Poder',
    'Unidade_Orcamentaria',
    'Recurso',
    'Fonte_Recurso',
    'Funcao',
    'Subfuncao',
    'Acao_Programatica',
    'Iniciativa',
    'Projeto',
    'Subprojeto',
    'Grupo_Despesa',
    'Modalidade',
    'Elemento',
    'Rubrica'
]


# Validar tipos

In [ ]:
fact_pagamentos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38210 entries, 0 to 38209
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Data                  38210 non-null  datetime64[ns]
 1   Fase_Gasto            38210 non-null  object        
 2   Empenho               38210 non-null  int64         
 3   CPF_CNPJ              38210 non-null  object        
 4   Credor                38210 non-null  object        
 5   Orgao                 38210 non-null  object        
 6   Valor                 38210 non-null  float64       
 7   Processo              38210 non-null  int64         
 8   Poder                 38210 non-null  object        
 9   Unidade_Orcamentaria  38210 non-null  object        
 10  Recurso               38210 non-null  object        
 11  Fonte_Recurso         38210 non-null  object        
 12  Funcao                38210 non-null  object        
 13  Subfuncao       

# Estatísticas

In [ ]:
fact_pagamentos['Valor'].describe()

,Valor
count,3.821000e+04
mean,2.149627e+05
std,4.507066e+06
min,-5.000000e+07
25%,3.015200e+02
50%,3.088000e+03
75%,2.958825e+04
max,7.313897e+08


## Verificação de Valores Negativos

Os registros serão analisados antes da definição dos indicadores financeiros.

In [ ]:
fact_pagamentos[
    fact_pagamentos['Valor'] < 0
].sort_values(
    'Valor'
)

,Data,Fase_Gasto,Empenho,CPF_CNPJ,Credor,Orgao,Valor,Processo,Poder,Unidade_Orcamentaria,...,Funcao,Subfuncao,Acao_Programatica,Iniciativa,Projeto,Subprojeto,Grupo_Despesa,Modalidade,Elemento,Rubrica
38209,2026-06-10,Pago,26003798046,00360305000104,Caixa Economica Federal,Secretaria de Habitacao e Regularizacao Fundiaria,-50000000.00,24170000005604,Poder Executivo,Fundo Estadual de Habitacao de Interesse Social,...,Habitacao,Habitacao Urbana,Acoes Habitacionais e Regularizacao Fundiaria,Promocao de Acoes Habitacionais,Porta de Entrada,Porta de Entrada Cidadao,Outras Despesas Correntes,Aplicacoes Diretas,Outros Auxilios Financeiros a Pessoas Fisicas,Subsidios Programa Porta de Entrada
38208,2024-12-31,Pago,24002874389,35990791000129,Visia Construcao Industrializada Ltda,Secretaria de Habitacao e Regularizacao Fundiaria,-47747422.29,24170000003326,Poder Executivo,Fundo Estadual de Habitacao de Interesse Social,...,Habitacao,Habitacao Urbana,Acoes Habitacionais e Regularizacao Fundiaria,Promocao de Acoes Habitacionais,Producao de Acoes Habitacionais,Producao de Acoes Habitacionais - Enfrentament...,Investimentos,Aplicacoes Diretas,Obras e Instalacoes,Modulos Habitacionais Temporarios
38207,2024-12-31,Pago,24003287639,92934215000106,Banrisul Solucoes Em Pagamentos S.a.inst Pagto,Secretaria de Desenvolvimento Social,-30608116.92,24210000011933,Poder Executivo,Gabinete e Orgaos Centrais.,...,Assistencia Social,Assistencia Comunitaria,Gestao Integrada Em Protecao e Defesa Civil,Auxilio Para Situacoes de Calamidade Ou Emerge...,Auxilio Emergencial Rs - Sedes,Auxilio a Familias Atingidas Por Eventos Clima...,Outras Despesas Correntes,Aplicacoes Diretas,Outros Auxilios Financeiros a Pessoas Fisicas,Assistencia Social a Pessoas
38206,2024-12-31,Pago,24004150093,92934215000106,Banrisul Solucoes Em Pagamentos S.a.inst Pagto,Secretaria de Desenvolvimento Social,-28257500.00,24210000014495,Poder Executivo,Gabinete e Orgaos Centrais.,...,Assistencia Social,Assistencia Comunitaria,Gestao Integrada Em Protecao e Defesa Civil,Auxilio Para Situacoes de Calamidade Ou Emerge...,Auxilio Emergencial Rs - Sedes,Auxilio a Familias Atingidas Por Eventos Clima...,Outras Despesas Correntes,Aplicacoes Diretas,Outros Auxilios Financeiros a Pessoas Fisicas,Assistencia Social a Pessoas
38205,2024-12-31,Pago,24006174953,92934215000106,Banrisul Solucoes Em Pagamentos S.a.inst Pagto,Secretaria de Desenvolvimento Social,-27547500.00,24210000018423,Poder Executivo,Gabinete e Orgaos Centrais.,...,Assistencia Social,Assistencia Comunitaria,Gestao Integrada Em Protecao e Defesa Civil,Auxilio Para Situacoes de Calamidade Ou Emerge...,Auxilio Emergencial Rs - Sedes,Auxilio a Familias Atingidas Por Eventos Clima...,Outras Despesas Correntes,Aplicacoes Diretas,Outros Auxilios Financeiros a Pessoas Fisicas,Assistencia Social a Pessoas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37732,2024-08-30,Pago,24004303426,627.***.***-87,Alessandro Antonio Monteiro,Fundacao Estadual de Protecao Ambiental Henriq...,-3.43,24056700004587,Poder Executivo,Fundacao Estadual de Protecao Ambiental Henriq...,...,Gestao Ambiental,Administracao Geral,"Gestao, Manutencao e Servicos Ao Estado - Fepam",Apoio Administrativo e Qualificacao da Infraes...,Apoio Administrativo e Qualificacao da Infraes...,Apoio Administrativo e Qualificacao da Infraes...,Outras Despesas Correntes,Aplicacoes Diretas,Outros Servicos de Terceiros-pessoa Juridica,Servicos Relacionados a Calamidade Publica Por...
37731,2024-07-30,Pago,24004123087,978.***.***-53,Barbara Anflor Pereira,Tribunal de Contas do Estado,-1.00,18790220248,Tribunal de Contas do Estado,Fundo de Reaparelhamento do Tribunal de Contas...,...,Legislativa,Controle Externo,"Gestao, Manutencao e Servicos Ao Estado - Tce",Apoio Administrativo e Qualificacao da Infraes...,Reaparelhamento e Modernizacao do Tce,Reaparelhamento e Modernizacao do Tce,Investimentos,Aplicacoes Diretas,Equipamentos e Material Permanente,"Ma

## Análise de Valores Negativos

Foram identificados registros financeiros com valores negativos.

Esses registros não representam necessariamente erros de qualidade dos dados.

Na execução orçamentária e financeira da administração pública, valores negativos podem indicar:

- estornos
- devoluções
- ajustes contábeis
- correções administrativas
- reclassificações financeiras

A presença desses lançamentos é esperada em bases financeiras governamentais.

---

## Decisão Metodológica

Os registros serão mantidos na base.

Dessa forma, os indicadores financeiros refletirão os valores efetivamente executados pelo Estado, considerando tanto pagamentos quanto ajustes posteriores.

# Valor total líquido

In [ ]:
fact_pagamentos['Valor'].sum()

np.float64(8213723318.379999)

# Valor total positivo

In [ ]:
fact_pagamentos[
    fact_pagamentos['Valor'] > 0
]['Valor'].sum()

np.float64(8666387894.18)

# Valor total negativo

In [ ]:
fact_pagamentos[
    fact_pagamentos['Valor'] < 0
]['Valor'].sum()

np.float64(-452664575.8000001)

# Número de registros negativos

In [ ]:
(
    fact_pagamentos['Valor'] < 0
).sum()

np.int64(482)

## Validação Financeira

Foi realizada a validação do valor financeiro consolidado da base de pagamentos.

### Resultados

- Registros analisados: 37.912
- Registros negativos: 472
- Valor líquido executado: R$ 7,97 bilhões

### Interpretação

Os registros negativos representam aproximadamente 1,24% da base e estão associados a:

- estornos
- ajustes contábeis
- correções administrativas
- reclassificações financeiras

Esses registros foram mantidos para preservar a consistência da execução financeira apresentada pelo Estado.

O valor líquido obtido é compatível com os totais divulgados no Painel de Transparência da Crise Climática do Rio Grande do Sul.

# Exportar CSV

In [ ]:
fact_pagamentos_bigquery = fact_pagamentos.copy()

# Identificadores como texto
for coluna in ["Empenho", "CPF_CNPJ", "Processo"]:
    fact_pagamentos_bigquery[coluna] = (
        fact_pagamentos_bigquery[coluna]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

# CPF/CNPJ apenas com dígitos
fact_pagamentos_bigquery["CPF_CNPJ"] = (
    fact_pagamentos_bigquery["CPF_CNPJ"]
    .str.replace(r"\D", "", regex=True)
)

# Valor numérico
fact_pagamentos_bigquery["Valor"] = pd.to_numeric(
    fact_pagamentos_bigquery["Valor"],
    errors="coerce"
)

arquivo_bigquery = os.path.join(
    pasta_raw,
    "FACT_PAGAMENTOS_BIGQUERY.csv"
)

fact_pagamentos_bigquery.to_csv(
    arquivo_bigquery,
    index=False,
    sep=",",
    encoding="utf-8"
)

print("✅ FACT_PAGAMENTOS_BIGQUERY.csv criado com sucesso.")

✅ FACT_PAGAMENTOS_BIGQUERY.csv criado com sucesso.


## Exportação Posterior

A `FACT_PAGAMENTOS` será exportada após a criação do campo `Documento_Original`.

Esse campo será utilizado como chave de relacionamento com a `DIM_CREDOR_CNPJ`.

A exportação posterior evita a geração de uma versão intermediária sem a chave cadastral.

# 9. Construção da DIM_CREDOR_CNPJ

## Objetivo

Construir uma dimensão de credores utilizando os documentos presentes na `FACT_PAGAMENTOS` e os dados públicos do Cadastro Nacional da Pessoa Jurídica (CNPJ).

A dimensão permitirá identificar informações cadastrais dos credores, incluindo nome fantasia, situação cadastral, atividade econômica, Unidade da Federação e município do estabelecimento.

O município identificado representa o endereço cadastral do credor e não necessariamente o município beneficiado ou o local onde o recurso foi aplicado.

---

## Tabela de Origem

DIM_CREDOR_CNPJ.csv

A tabela foi obtida através do cruzamento entre:

- Documentos presentes na `FACT_PAGAMENTOS`
- Base pública de estabelecimentos do CNPJ
- Diretório de municípios brasileiros

---

## Estratégia de Construção

A base de pagamentos possui 37.912 registros financeiros, mas apresenta repetição de documentos, pois um mesmo credor pode receber diversos pagamentos.

Os documentos foram consolidados para produzir uma única observação por credor.

A consulta original resultou em:

- 8.024 documentos únicos
- 2.981 CNPJs localizados
- 5.043 documentos sem correspondência na base do CNPJ
- 37,15% de cobertura cadastral

Após a padronização, um registro sem chave válida foi removido da dimensão, resultando em 8.023 documentos válidos.

Os documentos sem correspondência podem representar:

- Pessoas físicas
- Credores estrangeiros
- Identificadores especiais
- Documentos que não correspondem a um CNPJ
- Registros não identificados na fotografia cadastral utilizada

---

## Tratamento dos Documentos

Durante a carga inicial no BigQuery, os documentos foram interpretados como valores numéricos.

Esse processo removeu os zeros localizados no início de alguns CNPJs.

Exemplos:

- `360305000104` foi tratado como `00360305000104`
- `2885855000172` foi tratado como `02885855000172`
- `472805000138` foi tratado como `00472805000138`

Os documentos foram padronizados e posteriormente validados através da correspondência com a base pública do CNPJ.

---

## Granularidade

Cada registro representa um único documento identificado na base de pagamentos.

Um credor pode possuir diversos pagamentos na `FACT_PAGAMENTOS`, mas aparece somente uma vez na `DIM_CREDOR_CNPJ`.

Relacionamento:

`DIM_CREDOR_CNPJ` 1:N `FACT_PAGAMENTOS`

Chave de relacionamento:

`Documento_Original`

---

## Limitação Metodológica

O campo `Municipio_Credor` representa o município onde o estabelecimento está cadastrado na base do CNPJ.

Essa informação não comprova que o recurso tenha sido aplicado no mesmo município.

Uma empresa cadastrada em Porto Alegre, por exemplo, pode ter executado uma obra ou prestado um serviço em outro município.

Por esse motivo, serão utilizados os campos:

- Municipio_Credor
- Cod_IBGE_Credor
- UF_Credor

Não serão utilizados os termos:

- Municipio_Beneficiado
- Municipio_Repasse
- Municipio_Aplicacao_Recurso

# Carregamento da dimensão

In [ ]:
# Caminho do arquivo

arquivo_credor_cnpj = os.path.join(
    pasta_raw,
    "DIM_CREDOR_CNPJ.csv"
)

# Carregamento da dimensão

dim_credor_cnpj = pd.read_csv(
    arquivo_credor_cnpj,
    dtype={
        "Documento_Original": "string",
        "CNPJ": "string",
        "Cod_IBGE_Credor": "string",
        "CEP_Credor": "string",
        "CNAE_Principal": "string"
    }
)

# Remover possíveis espaços nos nomes das colunas

dim_credor_cnpj.columns = (
    dim_credor_cnpj.columns
    .str.strip()
)

print("DIM_CREDOR_CNPJ")
print("-" * 50)
print(f"Linhas : {dim_credor_cnpj.shape[0]}")
print(f"Colunas: {dim_credor_cnpj.shape[1]}")

display(dim_credor_cnpj.head())

DIM_CREDOR_CNPJ
--------------------------------------------------
Linhas : 8024
Colunas: 20


,Documento_Original,CNPJ,Credor,Nome_Fantasia,Situacao_Cadastral,Matriz_Filial,UF_Credor,Cod_IBGE_Credor,CEP_Credor,Bairro_Credor,Tipo_Logradouro_Credor,Logradouro_Credor,Numero_Endereco_Credor,CNAE_Principal,Quantidade_Pagamentos,Valor_Total_Pago,Data_Primeiro_Pagamento,Data_Ultimo_Pagamento,Data_Referencia_CNPJ,Status_Localizacao
0,46191353000117,46191353000117,Portos Rs,PORTOS RS,2.0,1.0,RS,4315602,96201020,GETULIO VARGAS,AVENIDA,HONORIO BICALHO,SN,5231101,1,7.313897e+08,2024-12-19,2024-12-19,2026-01-11,CNPJ localizado
1,92934215000106,92934215000106,Banrisul Solucoes Em Pagamentos S.a.inst Pagto,BANRISUL PAGAMENTOS,2.0,1.0,RS,4314902,90010000,CENTRO HISTORICO,RUA,SIQUEIRA CAMPOS,832,8299799,95,4.246816e+08,2024-05-15,2026-08-19,2026-01-11,CNPJ localizado
2,360305000104,00360305000104,Caixa Economica Federal,CEF MATRIZ,2.0,1.0,DF,5300108,70092900,ASA SUL,SETOR,SETOR SBS,S/N,6423900,102,3.039175e+08,2024-11-25,2026-08-12,2026-01-11,CNPJ localizado
3,92702067000196,92702067000196,Banco do Estado do Rio Grande do Sul S/a,BANRISUL,2.0,1.0,RS,4314902,90010040,CENTRO,RUA,CAPITAO MONTANHA,177,6422100,3,2.512522e+08,2024-07-23,2026-01-09,2026-01-11,CNPJ localizado
4,99999999999962,<NA>,Fbr Aviation Inc,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,45,2.447143e+08,2024-11-26,2026-08-04,2026-01-11,Identificador especial ou credor estrangeiro


# Verificação das colunas

In [ ]:
print("Colunas disponíveis:\n")

print(
    dim_credor_cnpj.columns.tolist()
)

Colunas disponíveis:

['Documento_Original', 'CNPJ', 'Credor', 'Nome_Fantasia', 'Situacao_Cadastral', 'Matriz_Filial', 'UF_Credor', 'Cod_IBGE_Credor', 'CEP_Credor', 'Bairro_Credor', 'Tipo_Logradouro_Credor', 'Logradouro_Credor', 'Numero_Endereco_Credor', 'CNAE_Principal', 'Quantidade_Pagamentos', 'Valor_Total_Pago', 'Data_Primeiro_Pagamento', 'Data_Ultimo_Pagamento', 'Data_Referencia_CNPJ', 'Status_Localizacao']


# Validação das colunas obrigatórias

In [ ]:
colunas_obrigatorias = [
    "Documento_Original",
    "CNPJ",
    "Credor",
    "Nome_Fantasia",
    "Situacao_Cadastral",
    "Matriz_Filial",
    "UF_Credor",
    "Cod_IBGE_Credor",
    "CEP_Credor",
    "Bairro_Credor",
    "Tipo_Logradouro_Credor",
    "Logradouro_Credor",
    "Numero_Endereco_Credor",
    "CNAE_Principal",
    "Quantidade_Pagamentos",
    "Valor_Total_Pago",
    "Data_Primeiro_Pagamento",
    "Data_Ultimo_Pagamento",
    "Data_Referencia_CNPJ",
    "Status_Localizacao"
]

colunas_ausentes = [
    coluna
    for coluna in colunas_obrigatorias
    if coluna not in dim_credor_cnpj.columns
]

if colunas_ausentes:
    raise ValueError(
        "Colunas ausentes na DIM_CREDOR_CNPJ: "
        + ", ".join(colunas_ausentes)
    )

print(
    "✅ Todas as colunas obrigatórias foram identificadas."
)

✅ Todas as colunas obrigatórias foram identificadas.


# Padronização do documento original

In [ ]:
dim_credor_cnpj["Documento_Original"] = (
    dim_credor_cnpj["Documento_Original"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.lstrip("0")
    .replace("", pd.NA)
)

# Padronização do CNPJ

In [ ]:
dim_credor_cnpj["CNPJ"] = (
    dim_credor_cnpj["CNPJ"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .replace("", pd.NA)
)

# Manter somente CNPJs com 14 dígitos

mascara_cnpj_invalido = (
    dim_credor_cnpj["CNPJ"].notna()
    & dim_credor_cnpj["CNPJ"].str.len().ne(14)
)

dim_credor_cnpj.loc[
    mascara_cnpj_invalido,
    "CNPJ"
] = pd.NA

# Conversão CNAE_Principal

In [ ]:
dim_credor_cnpj["CNAE_Principal"] = (
    dim_credor_cnpj["CNAE_Principal"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .replace("", pd.NA)
)

# Padronização do Código IBGE do credor

In [ ]:
dim_credor_cnpj["Cod_IBGE_Credor"] = (
    dim_credor_cnpj["Cod_IBGE_Credor"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .replace("", pd.NA)
)

# Padronização do CEP

In [ ]:
dim_credor_cnpj["CEP_Credor"] = (
    dim_credor_cnpj["CEP_Credor"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .replace("", pd.NA)
)

mascara_cep = (
    dim_credor_cnpj["CEP_Credor"]
    .notna()
)

dim_credor_cnpj.loc[
    mascara_cep,
    "CEP_Credor"
] = (
    dim_credor_cnpj.loc[
        mascara_cep,
        "CEP_Credor"
    ]
    .str.zfill(8)
)

# Padronização do CNAE principal

In [ ]:
dim_credor_cnpj["CNAE_Principal"] = (
    dim_credor_cnpj["CNAE_Principal"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .replace("", pd.NA)
)

# Conversão dos campos numéricos

In [ ]:
dim_credor_cnpj["Quantidade_Pagamentos"] = (
    pd.to_numeric(
        dim_credor_cnpj["Quantidade_Pagamentos"],
        errors="coerce"
    )
    .astype("Int64")
)

dim_credor_cnpj["Valor_Total_Pago"] = (
    pd.to_numeric(
        dim_credor_cnpj["Valor_Total_Pago"],
        errors="coerce"
    )
    .round(2)
)

# Conversão dos campos de data

In [ ]:
colunas_data_credor = [
    "Data_Primeiro_Pagamento",
    "Data_Ultimo_Pagamento",
    "Data_Referencia_CNPJ"
]

for coluna in colunas_data_credor:
    dim_credor_cnpj[coluna] = pd.to_datetime(
        dim_credor_cnpj[coluna],
        errors="coerce"
    )

# Verificação dos tipos

In [ ]:
print("Tipos após a conversão:\n")

display(
    dim_credor_cnpj[
        [
            "Documento_Original",
            "CNPJ",
            "Cod_IBGE_Credor",
            "CEP_Credor",
            "CNAE_Principal",
            "Quantidade_Pagamentos",
            "Valor_Total_Pago",
            "Data_Primeiro_Pagamento",
            "Data_Ultimo_Pagamento",
            "Data_Referencia_CNPJ"
        ]
    ].dtypes.to_frame("Tipo")
)

Tipos após a conversão:



,Tipo
Documento_Original,string[python]
CNPJ,string[python]
Cod_IBGE_Credor,string[python]
CEP_Credor,string[python]
CNAE_Principal,string[python]
Quantidade_Pagamentos,Int64
Valor_Total_Pago,float64
Data_Primeiro_Pagamento,datetime64[ns]
Data_Ultimo_Pagamento,datetime64[ns]
Data_Referencia_CNPJ,datetime64[ns]


# Verificação da Estrutura

In [ ]:
dim_credor_cnpj.info()

display(
    dim_credor_cnpj.head()
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8024 entries, 0 to 8023
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Documento_Original       8023 non-null   string        
 1   CNPJ                     2981 non-null   string        
 2   Credor                   8024 non-null   object        
 3   Nome_Fantasia            2278 non-null   object        
 4   Situacao_Cadastral       2981 non-null   float64       
 5   Matriz_Filial            2981 non-null   float64       
 6   UF_Credor                2981 non-null   object        
 7   Cod_IBGE_Credor          2980 non-null   string        
 8   CEP_Credor               2980 non-null   string        
 9   Bairro_Credor            2964 non-null   object        
 10  Tipo_Logradouro_Credor   2966 non-null   object        
 11  Logradouro_Credor        2981 non-null   object        
 12  Numero_Endereco_Credor   2981 non-

,Documento_Original,CNPJ,Credor,Nome_Fantasia,Situacao_Cadastral,Matriz_Filial,UF_Credor,Cod_IBGE_Credor,CEP_Credor,Bairro_Credor,Tipo_Logradouro_Credor,Logradouro_Credor,Numero_Endereco_Credor,CNAE_Principal,Quantidade_Pagamentos,Valor_Total_Pago,Data_Primeiro_Pagamento,Data_Ultimo_Pagamento,Data_Referencia_CNPJ,Status_Localizacao
0,46191353000117,46191353000117,Portos Rs,PORTOS RS,2.0,1.0,RS,4315602,96201020,GETULIO VARGAS,AVENIDA,HONORIO BICALHO,SN,5231101,1,7.313897e+08,2024-12-19,2024-12-19,2026-01-11,CNPJ localizado
1,92934215000106,92934215000106,Banrisul Solucoes Em Pagamentos S.a.inst Pagto,BANRISUL PAGAMENTOS,2.0,1.0,RS,4314902,90010000,CENTRO HISTORICO,RUA,SIQUEIRA CAMPOS,832,8299799,95,4.246816e+08,2024-05-15,2026-08-19,2026-01-11,CNPJ localizado
2,360305000104,00360305000104,Caixa Economica Federal,CEF MATRIZ,2.0,1.0,DF,5300108,70092900,ASA SUL,SETOR,SETOR SBS,S/N,6423900,102,3.039175e+08,2024-11-25,2026-08-12,2026-01-11,CNPJ localizado
3,92702067000196,92702067000196,Banco do Estado do Rio Grande do Sul S/a,BANRISUL,2.0,1.0,RS,4314902,90010040,CENTRO,RUA,CAPITAO MONTANHA,177,6422100,3,2.512522e+08,2024-07-23,2026-01-09,2026-01-11,CNPJ localizado
4,99999999999962,<NA>,Fbr Aviation Inc,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,<NA>,45,2.447143e+08,2024-11-26,2026-08-04,2026-01-11,Identificador especial ou credor estrangeiro


# Remoção de registros sem chave

In [ ]:
print(
    "Registros antes da remoção:",
    len(dim_credor_cnpj)
)

dim_credor_cnpj = (
    dim_credor_cnpj
    .dropna(
        subset=["Documento_Original"]
    )
    .copy()
)

print(
    "Registros após a remoção:",
    len(dim_credor_cnpj)
)

Registros antes da remoção: 8024
Registros após a remoção: 8023


# Verificação de Duplicidades

In [ ]:
duplicados_credor = (
    dim_credor_cnpj["Documento_Original"]
    .duplicated()
    .sum()
)

print(
    "Documentos duplicados:",
    duplicados_credor
)

print(
    "Documentos nulos:",
    dim_credor_cnpj[
        "Documento_Original"
    ].isna().sum()
)

print(
    "Documentos únicos:",
    dim_credor_cnpj[
        "Documento_Original"
    ].nunique()
)

Documentos duplicados: 0
Documentos nulos: 0
Documentos únicos: 8023


# Validação da granularidade

In [ ]:
if duplicados_credor > 0:
    raise ValueError(
        "A DIM_CREDOR_CNPJ possui documentos duplicados."
    )

if dim_credor_cnpj["Documento_Original"].isna().any():
    raise ValueError(
        "A DIM_CREDOR_CNPJ possui documentos nulos."
    )

print(
    "✅ Granularidade validada: "
    "um registro por Documento_Original."
)

✅ Granularidade validada: um registro por Documento_Original.


# Verificação de Valores Nulos

In [ ]:
nulos_credor = pd.DataFrame({
    "Qtd_Nulos": (
        dim_credor_cnpj
        .isnull()
        .sum()
    ),
    "Perc_Nulos": (
        dim_credor_cnpj
        .isnull()
        .mean()
        .mul(100)
        .round(2)
    )
})

display(
    nulos_credor.sort_values(
        "Perc_Nulos",
        ascending=False
    )
)

,Qtd_Nulos,Perc_Nulos
Nome_Fantasia,5745,71.61
Bairro_Credor,5059,63.06
Tipo_Logradouro_Credor,5057,63.03
Cod_IBGE_Credor,5043,62.86
CEP_Credor,5043,62.86
Numero_Endereco_Credor,5042,62.84
CNPJ,5042,62.84
CNAE_Principal,5042,62.84
UF_Credor,5042,62.84
Matriz_Filial,5042,62.84


# Validação da Cobertura Cadastral

In [ ]:
resumo_localizacao = (
    dim_credor_cnpj[
        "Status_Localizacao"
    ]
    .value_counts(dropna=False)
    .rename_axis("Status_Localizacao")
    .reset_index(name="Quantidade")
)

resumo_localizacao["Percentual"] = (
    resumo_localizacao["Quantidade"]
    .div(len(dim_credor_cnpj))
    .mul(100)
    .round(2)
)

display(resumo_localizacao)

,Status_Localizacao,Quantidade,Percentual
0,Documento sem correspondência na base CNPJ,5041,62.83
1,CNPJ localizado,2981,37.16
2,Identificador especial ou credor estrangeiro,1,0.01


# Quantidade de CNPJs Localizados

In [ ]:
total_documentos = len(
    dim_credor_cnpj
)

total_localizados = (
    dim_credor_cnpj["CNPJ"]
    .notna()
    .sum()
)

total_nao_localizados = (
    dim_credor_cnpj["CNPJ"]
    .isna()
    .sum()
)

percentual_localizado = round(
    total_localizados
    / total_documentos
    * 100,
    2
)

print(
    f"Documentos válidos: "
    f"{total_documentos}"
)

print(
    f"CNPJs localizados: "
    f"{total_localizados}"
)

print(
    f"Documentos não localizados: "
    f"{total_nao_localizados}"
)

print(
    f"Cobertura cadastral: "
    f"{percentual_localizado}%"
)

Documentos válidos: 8023
CNPJs localizados: 2981
Documentos não localizados: 5042
Cobertura cadastral: 37.16%


# Distribuição dos Credores por UF

In [ ]:
credores_por_uf = (
    dim_credor_cnpj
    .dropna(
        subset=["UF_Credor"]
    )
    .groupby(
        "UF_Credor",
        as_index=False
    )
    .agg(
        Quantidade_Credores=(
            "Documento_Original",
            "nunique"
        ),
        Quantidade_Pagamentos=(
            "Quantidade_Pagamentos",
            "sum"
        ),
        Valor_Total_Pago=(
            "Valor_Total_Pago",
            "sum"
        )
    )
    .sort_values(
        "Valor_Total_Pago",
        ascending=False
    )
)

credores_por_uf["Valor_Total_Pago"] = (
    credores_por_uf["Valor_Total_Pago"]
    .round(2)
)

display(
    credores_por_uf.head(10)
)

,UF_Credor,Quantidade_Credores,Quantidade_Pagamentos,Valor_Total_Pago
19,RS,2659,10321,6.031899e+09
22,SP,90,575,6.224596e+08
4,DF,15,197,4.050036e+08
20,SC,51,258,1.153330e+08
9,MG,34,155,1.119480e+08
7,GO,12,42,6.252197e+07
15,PR,52,178,3.128849e+07
1,AM,2,10,1.934020e+07
5,ES,18,77,1.565956e+07
3,CE,6,77,3.309579e+06


# Inclusão do município cadastral dos credores do RS

In [ ]:
# Criar mapa dos municípios do Rio Grande do Sul

mapa_municipios_rs = (
    dim_municipio[
        [
            "Cod_IBGE",
            "Municipio"
        ]
    ]
    .copy()
)

# Padronizar o Código IBGE

mapa_municipios_rs["Cod_IBGE"] = (
    mapa_municipios_rs["Cod_IBGE"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

dim_credor_cnpj["Cod_IBGE_Credor"] = (
    dim_credor_cnpj["Cod_IBGE_Credor"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

# Renomear os campos do mapa

mapa_municipios_rs = (
    mapa_municipios_rs
    .rename(
        columns={
            "Cod_IBGE": "Cod_IBGE_Credor",
            "Municipio": "Municipio_Credor"
        }
    )
)

# Adicionar o município à dimensão

dim_credor_cnpj = dim_credor_cnpj.merge(
    mapa_municipios_rs,
    on="Cod_IBGE_Credor",
    how="left",
    validate="many_to_one"
)

print(
    "Credores com município identificado:",
    dim_credor_cnpj[
        "Municipio_Credor"
    ].notna().sum()
)

print(
    "Credores sem município identificado:",
    dim_credor_cnpj[
        "Municipio_Credor"
    ].isna().sum()
)

Credores com município identificado: 2659
Credores sem município identificado: 5364


# Principais Municípios Cadastrais dos Credores do RS

In [ ]:
credores_por_municipio = (
    dim_credor_cnpj[
        dim_credor_cnpj[
            "UF_Credor"
        ].eq("RS")
    ]
    .dropna(
        subset=[
            "Cod_IBGE_Credor",
            "Municipio_Credor"
        ]
    )
    .groupby(
        [
            "Cod_IBGE_Credor",
            "Municipio_Credor",
            "UF_Credor"
        ],
        as_index=False
    )
    .agg(
        Quantidade_Credores=(
            "Documento_Original",
            "nunique"
        ),
        Quantidade_Pagamentos=(
            "Quantidade_Pagamentos",
            "sum"
        ),
        Valor_Total_Pago=(
            "Valor_Total_Pago",
            "sum"
        )
    )
    .sort_values(
        "Valor_Total_Pago",
        ascending=False
    )
)

credores_por_municipio[
    "Valor_Total_Pago"
] = (
    credores_por_municipio[
        "Valor_Total_Pago"
    ]
    .round(2)
)

display(
    credores_por_municipio.head(20)
)

,Cod_IBGE_Credor,Municipio_Credor,UF_Credor,Quantidade_Credores,Quantidade_Pagamentos,Valor_Total_Pago
325,4314902,Porto Alegre,RS,309,2795,2.758003e+09
343,4315602,Rio Grande,RS,15,28,7.481213e+08
364,4316907,Santa Maria,RS,25,683,2.851767e+08
79,4304606,Canoas,RS,35,278,1.759586e+08
398,4318705,São Leopoldo,RS,13,208,1.579660e+08
5,4300406,Alegrete,RS,6,79,1.282404e+08
363,4316808,Santa Cruz do Sul,RS,24,55,1.266930e+08
95,4305108,Caxias do Sul,RS,31,181,1.212028e+08
231,4311403,Lajeado,RS,27,177,1.058859e+08
217,4310801,Ivoti,RS,9,54,1.057737e+08


# Validação dos municípios cadastrais

In [ ]:
print(
    "Municípios com credores cadastrados:",
    credores_por_municipio[
        "Cod_IBGE_Credor"
    ].nunique()
)

print(
    "Credores localizados nesses municípios:",
    credores_por_municipio[
        "Quantidade_Credores"
    ].sum()
)

print(
    "Pagamentos associados:",
    credores_por_municipio[
        "Quantidade_Pagamentos"
    ].sum()
)

print(
    "Valor total associado: "
    f"R$ "
    f"{credores_por_municipio['Valor_Total_Pago'].sum():,.2f}"
)

Municípios com credores cadastrados: 497
Credores localizados nesses municípios: 2659
Pagamentos associados: 10321
Valor total associado: R$ 6,031,898,661.87


# Criação da chave na FACT_PAGAMENTOS

In [ ]:
fact_pagamentos["Documento_Original"] = (
    fact_pagamentos["CPF_CNPJ"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.lstrip("0")
    .replace("", pd.NA)
)

print(
    "Pagamentos com documento:",
    fact_pagamentos[
        "Documento_Original"
    ].notna().sum()
)

print(
    "Pagamentos sem documento:",
    fact_pagamentos[
        "Documento_Original"
    ].isna().sum()
)

print(
    "Documentos únicos na FACT:",
    fact_pagamentos[
        "Documento_Original"
    ].nunique()
)

Pagamentos com documento: 38200
Pagamentos sem documento: 10
Documentos únicos na FACT: 8040


# Validação do relacionamento

In [ ]:
validacao_credor = (
    fact_pagamentos[
        [
            "Documento_Original",
            "Valor"
        ]
    ]
    .merge(
        dim_credor_cnpj[
            [
                "Documento_Original",
                "Status_Localizacao",
                "Municipio_Credor",
                "UF_Credor"
            ]
        ],
        on="Documento_Original",
        how="left",
        validate="many_to_one"
    )
)

pagamentos_sem_dimensao = (
    validacao_credor[
        "Status_Localizacao"
    ]
    .isna()
    .sum()
)

print(
    "Pagamentos sem correspondência "
    "na DIM_CREDOR_CNPJ:",
    pagamentos_sem_dimensao
)

Pagamentos sem correspondência na DIM_CREDOR_CNPJ: 31


# Cobertura financeira por localização

In [ ]:
resumo_financeiro_localizacao = (
    validacao_credor
    .groupby(
        "Status_Localizacao",
        dropna=False
    )
    .agg(
        Quantidade_Pagamentos=(
            "Valor",
            "count"
        ),
        Valor_Total=(
            "Valor",
            "sum"
        )
    )
    .reset_index()
)

resumo_financeiro_localizacao[
    "Valor_Total"
] = (
    resumo_financeiro_localizacao[
        "Valor_Total"
    ]
    .round(2)
)

display(
    resumo_financeiro_localizacao
)

,Status_Localizacao,Quantidade_Pagamentos,Valor_Total
0,CNPJ localizado,12284,7.659819e+09
1,Documento sem correspondência na base CNPJ,25848,3.046142e+08
2,Identificador especial ou credor estrangeiro,47,2.447773e+08
3,NaN,31,4.512933e+06


# Validação da preservação financeira

In [ ]:
valor_total_fact = round(
    fact_pagamentos["Valor"].sum(),
    2
)

valor_total_validacao = round(
    validacao_credor["Valor"].sum(),
    2
)

print(
    f"Valor total na FACT_PAGAMENTOS: "
    f"R$ {valor_total_fact:,.2f}"
)

print(
    f"Valor total após a validação: "
    f"R$ {valor_total_validacao:,.2f}"
)

print(
    "Diferença financeira:",
    round(
        valor_total_fact
        - valor_total_validacao,
        2
    )
)

Valor total na FACT_PAGAMENTOS: R$ 8,213,723,318.38
Valor total após a validação: R$ 8,213,723,318.38
Diferença financeira: 0.0


# Renomear o campo de Credor

In [ ]:
# Renomear o campo de credor na dimensão
# para manter somente Documento_Original como chave

dim_credor_cnpj = dim_credor_cnpj.rename(
    columns={
        "Credor":
        "Credor_Cadastral"
    }
)


# Estrutura final da dimensão

In [ ]:
colunas_dim_credor = [
    "Documento_Original",
    "CNPJ",
    "Credor_Cadastral",
    "Nome_Fantasia",
    "Situacao_Cadastral",
    "Matriz_Filial",
    "UF_Credor",
    "Cod_IBGE_Credor",
    "Municipio_Credor",
    "CEP_Credor",
    "Bairro_Credor",
    "Tipo_Logradouro_Credor",
    "Logradouro_Credor",
    "Numero_Endereco_Credor",
    "CNAE_Principal",
    "Quantidade_Pagamentos",
    "Valor_Total_Pago",
    "Data_Primeiro_Pagamento",
    "Data_Ultimo_Pagamento",
    "Data_Referencia_CNPJ",
    "Status_Localizacao"
]

dim_credor_cnpj = (
    dim_credor_cnpj[
        colunas_dim_credor
    ]
    .copy()
)

# Validação final da dimensão

In [ ]:
print("Validação Final da DIM_CREDOR_CNPJ")
print("-" * 50)

print(
    f"Registros: "
    f"{len(dim_credor_cnpj)}"
)

print(
    f"Documentos únicos: "
    f"{dim_credor_cnpj['Documento_Original'].nunique()}"
)

print(
    f"CNPJs localizados: "
    f"{dim_credor_cnpj['CNPJ'].notna().sum()}"
)

print(
    f"Documentos duplicados: "
    f"{dim_credor_cnpj['Documento_Original'].duplicated().sum()}"
)

print(
    f"Documentos nulos: "
    f"{dim_credor_cnpj['Documento_Original'].isna().sum()}"
)

print(
    f"Valor total consolidado: "
    f"R$ {dim_credor_cnpj['Valor_Total_Pago'].sum():,.2f}"
)

Validação Final da DIM_CREDOR_CNPJ
--------------------------------------------------
Registros: 8023
Documentos únicos: 8023
CNPJs localizados: 2981
Documentos duplicados: 0
Documentos nulos: 0
Valor total consolidado: R$ 7,974,417,680.30


# Exportação da DIM_CREDOR_CNPJ

In [ ]:
arquivo_saida_dim_credor = os.path.join(
    pasta_clean,
    "DIM_CREDOR_CNPJ.xlsx"
)

dim_credor_cnpj.to_excel(
    arquivo_saida_dim_credor,
    index=False
)

print(
    "✅ DIM_CREDOR_CNPJ.xlsx criado com sucesso."
)

✅ DIM_CREDOR_CNPJ.xlsx criado com sucesso.


# Exportação de FACT_PAGAMENTOS

# 10. Construção da DIM_INDICADORES_SOCIAIS_RS

## Objetivo

Construir uma dimensão de indicadores sociais e econômicos do Rio Grande do Sul utilizando a base do Atlas do Desenvolvimento Humano (ADH).

Esta dimensão permitirá complementar as análises do projeto através de indicadores de desenvolvimento humano, educação, renda, desigualdade social, pobreza e expectativa de vida do estado.

---

## Tabela de Origem

adh_radar_base_2012_2024.xlsx

---

## Filtros Aplicados

### Recorte Geográfico

- Rio Grande do Sul

### Nível Territorial

- Unidade da Federação

### Ano de Referência

- 2024

---

## Indicadores Selecionados

### Identificação

- ANO
- CODIGO
- NOME

---

### Desenvolvimento Humano

- IDHM
- IDHM_L
- IDHM_E
- IDHM_R

---

### Condições Sociais e Econômicas

- RDPC (Renda Domiciliar per Capita)
- RENOCUP (Renda dos Ocupados)
- GINI
- THEIL

---

### Educação

- ANOSEST
- T_ANALF15M
- T_SUPER25M

---

### Saúde

- ESPVIDA
- MORT1

---

### Pobreza e Vulnerabilidade

- PIND
- PMPOB
- PPOB

---

### Demografia

- POP

---

## Aplicações Analíticas

A dimensão será utilizada para responder perguntas como:

- Qual o contexto socioeconômico do Rio Grande do Sul em 2024?
- Qual o nível de desenvolvimento humano do estado?
- Como renda, educação e desigualdade podem influenciar a capacidade de recuperação após eventos climáticos?
- Existe relação entre indicadores sociais e distribuição dos investimentos realizados?
- Como os indicadores estaduais se relacionam com os impactos observados nos municípios?

---

## Resultado Esperado

DIM_INDICADORES_SOCIAIS_RS

Campos principais:

- ANO
- CODIGO
- NOME
- IDHM
- IDHM_L
- IDHM_E
- IDHM_R
- ESPVIDA
- MORT1
- ANOSEST
- T_ANALF15M
- T_SUPER25M
- RDPC
- RENOCUP
- GINI
- THEIL
- PIND
- PMPOB
- PPOB
- POP

In [ ]:
arquivo_saida_fact_pagamentos = os.path.join(
    pasta_clean,
    "FACT_PAGAMENTOS.xlsx"
)

fact_pagamentos.to_excel(
    arquivo_saida_fact_pagamentos,
    index=False
)

print(
    "✅ FACT_PAGAMENTOS.xlsx atualizado "
    "com o campo Documento_Original."
)

✅ FACT_PAGAMENTOS.xlsx atualizado com o campo Documento_Original.


# Validação dos arquivos exportados

In [ ]:
arquivos_exportados = [
    "DIM_CREDOR_CNPJ.xlsx",
    "FACT_PAGAMENTOS.xlsx"
]

print("Validação das exportações:\n")

for arquivo in arquivos_exportados:

    caminho = os.path.join(
        pasta_clean,
        arquivo
    )

    if os.path.exists(caminho):
        tamanho_mb = (
            os.path.getsize(caminho)
            / 1024
            / 1024
        )

        print(
            f"✅ Arquivo criado: "
            f"{arquivo} "
            f"({tamanho_mb:.2f} MB)"
        )

    else:
        print(
            f"❌ Arquivo não encontrado: "
            f"{arquivo}"
        )

Validação das exportações:

✅ Arquivo criado: DIM_CREDOR_CNPJ.xlsx (0.85 MB)
✅ Arquivo criado: FACT_PAGAMENTOS.xlsx (5.11 MB)


# Filtrar Unidade da Federação Rio Grande do Sul - Ano 2024

In [ ]:
arquivo_adh = os.path.join(
    pasta_raw,
    "adh_radar_base_2012_2024.xlsx"
)

df_adh = pd.read_excel(arquivo_adh)

df_vuln = df_adh[
    (df_adh["ANO"] == 2024) &
    (df_adh["AGREGACAO"] == "Unidade da Federação") &
    (df_adh["NOME"] == "Rio Grande do Sul")
].copy()

# Seleção dos Campos

In [ ]:
colunas = [
    "ANO",
    "CODIGO",
    "NOME",
    "IDHM",
    "IDHM_L",
    "IDHM_E",
    "IDHM_R",
    "ANOSEST",
    "T_ANALF15M",
    "T_SUPER25M",
    "ESPVIDA",
    "MORT1",
    "RDPC",
    "RENOCUP",
    "GINI",
    "THEIL",
    "PIND",
    "PMPOB",
    "PPOB",
    "POP"
]

df_vuln = df_vuln[colunas].copy()

# Renomeação das Colunas

In [ ]:
df_vuln.rename(columns={
    "ANO": "Ano",
    "CODIGO": "Cod_UF",
    "NOME": "Nome_UF",
    "ANOSEST": "Escolaridade_Media",
    "T_ANALF15M": "Taxa_Analfabetismo",
    "T_SUPER25M": "Ensino_Superior",
    "ESPVIDA": "Expectativa_Vida",
    "MORT1": "Mortalidade_Infantil",
    "RDPC": "Renda_Per_Capita",
    "RENOCUP": "Renda_Ocupacao",
    "GINI": "Indice_Gini",
    "THEIL": "Indice_Theil",
    "PIND": "Perc_Indigencia",
    "PMPOB": "Perc_Pobreza_Moderada",
    "PPOB": "Perc_Pobreza",
    "POP": "Populacao"
}, inplace=True)

# Verificação de Nulos

In [ ]:
nulos = pd.DataFrame({
    "Qtd_Nulos": df_vuln.isnull().sum(),
    "%_Nulos": round(df_vuln.isnull().mean()*100,2)
})

nulos.sort_values("%_Nulos", ascending=False)

,Qtd_Nulos,%_Nulos
Ano,0,0.0
Cod_UF,0,0.0
Nome_UF,0,0.0
IDHM,0,0.0
IDHM_L,0,0.0
IDHM_E,0,0.0
IDHM_R,0,0.0
Escolaridade_Media,0,0.0
Taxa_Analfabetismo,0,0.0
Ensino_Superior,0,0.0


# Estrutura

In [ ]:
df_vuln.info()
df_vuln.head()

<class 'pandas.core.frame.DataFrame'>
Index: 1 entries, 424 to 424
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Ano                    1 non-null      int64  
 1   Cod_UF                 1 non-null      float64
 2   Nome_UF                1 non-null      object 
 3   IDHM                   1 non-null      float64
 4   IDHM_L                 1 non-null      float64
 5   IDHM_E                 1 non-null      float64
 6   IDHM_R                 1 non-null      float64
 7   Escolaridade_Media     1 non-null      float64
 8   Taxa_Analfabetismo     1 non-null      float64
 9   Ensino_Superior        1 non-null      float64
 10  Expectativa_Vida       1 non-null      float64
 11  Mortalidade_Infantil   1 non-null      float64
 12  Renda_Per_Capita       1 non-null      float64
 13  Renda_Ocupacao         1 non-null      float64
 14  Indice_Gini            1 non-null      float64
 15  Indice_Thei

,Ano,Cod_UF,Nome_UF,IDHM,IDHM_L,IDHM_E,IDHM_R,Escolaridade_Media,Taxa_Analfabetismo,Ensino_Superior,Expectativa_Vida,Mortalidade_Infantil,Renda_Per_Capita,Renda_Ocupacao,Indice_Gini,Indice_Theil,Perc_Indigencia,Perc_Pobreza_Moderada,Perc_Pobreza,Populacao
424,2024,43.0,Rio Grande do Sul,0.818,0.869,0.791,0.796,10.47,2.26,19.7,77.16,9.45,1132.46,1644.07,0.46,0.358,1.01,2.29,7.59,11558291


# Estatísticas Básicas

In [ ]:
df_vuln[
    [
        "IDHM",
        "Renda_Per_Capita",
        "Indice_Gini",
        "Expectativa_Vida"
    ]
].describe()

,IDHM,Renda_Per_Capita,Indice_Gini,Expectativa_Vida
count,1.000,1.00,1.00,1.00
mean,0.818,1132.46,0.46,77.16
std,NaN,NaN,NaN,NaN
min,0.818,1132.46,0.46,77.16
25%,0.818,1132.46,0.46,77.16
50%,0.818,1132.46,0.46,77.16
75%,0.818,1132.46,0.46,77.16
max,0.818,1132.46,0.46,77.16


# Cobertura

In [ ]:
print("Quantidade de registros:")
print(df_vuln.shape[0])
print(df_vuln)

Quantidade de registros:
1
      Ano  Cod_UF            Nome_UF   IDHM  IDHM_L  IDHM_E  IDHM_R  \
424  2024    43.0  Rio Grande do Sul  0.818   0.869   0.791   0.796   

     Escolaridade_Media  Taxa_Analfabetismo  Ensino_Superior  \
424               10.47                2.26             19.7   

     Expectativa_Vida  Mortalidade_Infantil  Renda_Per_Capita  Renda_Ocupacao  \
424             77.16                  9.45           1132.46         1644.07   

     Indice_Gini  Indice_Theil  Perc_Indigencia  Perc_Pobreza_Moderada  \
424         0.46         0.358             1.01                   2.29   

     Perc_Pobreza  Populacao  
424          7.59   11558291  


# Exportação

In [ ]:
arquivo_saida = os.path.join(
    pasta_clean,
    'DIM_INDICADORES_SOCIAIS_RS.xlsx'
)

df_vuln.to_excel(
    arquivo_saida,
    index=False
)

print(
    "✅ DIM_INDICADORES_SOCIAIS_RS.xlsx criado com sucesso."
)

✅ DIM_INDICADORES_SOCIAIS_RS.xlsx criado com sucesso.


# 11. Construção da FACT_REPASSE_MUNICIPIO

## Objetivo

Construir uma tabela fato com os pagamentos e repasses territorializados disponibilizados no Painel da Transparência da Crise Climática do Rio Grande do Sul.

A base contém o município associado ao pagamento ou repasse, permitindo analisar a distribuição territorial dos recursos relacionados às ações de resposta, recuperação e reconstrução.

Diferentemente do campo `Municipio_Credor`, que representa o endereço cadastral do recebedor, o campo `Municipio_Repasse` representa o município ao qual o Painel da Transparência associou o gasto ou o repasse.

---

## Tabela de Origem

REPASSE_MUNICIPIO.csv

Fonte:

Painel da Transparência da Crise Climática do Rio Grande do Sul.

O arquivo contém informações sobre:

- Órgão executor
- Região COREDE
- Município associado
- Situação do município
- Fonte do recurso
- Elemento da despesa
- Rubrica
- Projeto
- Subprojeto
- Data
- Fato contábil
- Unidade orçamentária
- Credor
- Função
- Número do empenho
- Valor pago ou repassado

---

## Estrutura da Base Original

A base possui:

- 5.801 registros
- 16 campos originais
- 4.984 números de empenho únicos
- Registros entre maio de 2024 e agosto de 2026
- Municípios classificados como calamidade, emergência, não homologados ou indefinidos

Um mesmo empenho pode aparecer em mais de um registro, pois determinado pagamento pode possuir diferentes datas, municípios, projetos, subprojetos ou parcelas financeiras.

Por esse motivo, os registros não serão consolidados ou eliminados apenas pela repetição do número do empenho.

---

## Granularidade

Cada registro representa uma ocorrência de pagamento ou repasse territorializado conforme apresentado no Painel da Transparência.

O número de empenho não será utilizado como chave única, pois um mesmo empenho pode aparecer em diferentes registros.

Será criada uma chave técnica chamada:

`ID_Repasse_Municipio`

---

## Tratamento do Município

Os registros territoriais possuem valores como:

- Porto Alegre
- Canoas
- Eldorado do Sul
- Santa Maria
- Município Indefinido
- A Classificar

Os municípios válidos serão relacionados com a `DIM_MUNICIPIO` através do nome normalizado.

Os registros classificados como município indefinido ou a classificar serão preservados na tabela, mas permanecerão sem Código IBGE.

Não será realizada distribuição artificial dos valores indefinidos entre os municípios.

---

## Relacionamento

A relação territorial será realizada pelo campo:

`Cod_IBGE`

Estrutura esperada:

```text
DIM_MUNICIPIO
        │
        │ Cod_IBGE
        │ 1:N
        ▼
FACT_REPASSE_MUNICIPIO
```

A mesma dimensão também se relacionará com a tabela de impacto:

```text
DIM_MUNICIPIO
        │
        │ Cod_IBGE
        ├─────────────────────────────┐
        ▼                             ▼
FACT_IMPACTO_MUNICIPIO      FACT_REPASSE_MUNICIPIO
```

---

## Aplicações Analíticas

A tabela permitirá responder perguntas como:

- Quanto foi pago ou repassado por município?
- Quais municípios concentraram os maiores valores?
- Quantos pagamentos ou repasses foram associados a cada município?
- Qual foi a distribuição por Região COREDE?
- Municípios em situação de calamidade receberam mais recursos?
- Municípios com maior população afetada receberam maiores valores?
- Qual percentual dos recursos possui município identificado?
- Qual percentual permanece classificado como município indefinido?
- Qual foi a evolução temporal dos recursos por município?
- Quais funções, projetos e subprojetos concentraram recursos em cada território?

---

## Limitação Metodológica

O campo `Municipio_Repasse` deve ser interpretado como o município associado ao registro pelo Painel da Transparência.

O campo não significa necessariamente que o valor foi transferido diretamente para a prefeitura.

A base inclui diferentes formas de execução, como:

- Transferências para fundos municipais
- Convênios
- Pagamentos a empresas por obras ou serviços executados no município
- Aquisição de bens
- Auxílios
- Investimentos estaduais
- Repasses para entidades públicas ou privadas

Por esse motivo, o indicador deverá ser descrito como:

`Valor pago ou repassado associado ao município`

Não deverá ser descrito genericamente como:

`Valor recebido pela prefeitura`

# Carregamento da base

In [ ]:
# Caminho do arquivo

arquivo_repasse_municipio = os.path.join(
    pasta_raw,
    "REPASSE_MUNICIPIO.csv"
)

# Carregamento da base

repasse_municipio = pd.read_csv(
    arquivo_repasse_municipio,
    sep=";",
    encoding="utf-8",
    dtype="string"
)

# Remover possíveis espaços nos nomes das colunas

repasse_municipio.columns = (
    repasse_municipio.columns
    .str.strip()
)

print("REPASSE_MUNICIPIO")
print("-" * 50)
print(f"Linhas : {repasse_municipio.shape[0]}")
print(f"Colunas: {repasse_municipio.shape[1]}")

display(
    repasse_municipio.head()
)

REPASSE_MUNICIPIO
--------------------------------------------------
Linhas : 5868
Colunas: 16


,Órgão,Região COREDE,Municipio,Situação Municipio,Recurso,Elemento,Rubrica,Projeto,Subprojeto,Data,Fato Contábil,Unidade Orçamentária,Credor,Função,Número Empenho,Pago/Repassado
0,18 - SEC LOG E TRANSPORTES,INDEFINIDO,Município Indefinido,Munícipio Indefinido,0110 - FUNRIGS PARCELAS DIVIDA,65 - CONSTITUICAO OU AUMENTO DE CAPITAL DE EMP...,6503 - PARTICIPACAO EM CONSTITUICAO OU AUMENTO...,8107 - AUM CAP VINC - PORTOS RS,8107.01.001 - AUMENTO DE CAPITAL - PORTOS RS,19/12/2024,0053 - PARTICIPACOES SOCIETARIAS EM EMPRESAS S...,1801 - GABINETE E ORGAOS CENTRAI,PORTOS RS,26 - TRANSPORTE,24007402766,"R$ 731.389.734,00"
1,12 - SEC. DA SEGURANCA PUBLICA,INDEFINIDO,Município Indefinido,Munícipio Indefinido,0110 - FUNRIGS PARCELAS DIVIDA,52 - EQUIPAMENTOS E MATERIAL PERMANENTE,5226 - AERONAVES E/OU EQUIPAMENTOS PARA AERONAVES,5876 - QUALIF INSTAL E SV BM,5876.01.016 - AQUISCAO DE AERONAVES - ENFRENTA...,28/5/2025,0040 - FORNECIMENTO DE BENS E/OU SERVICOS - NA...,1203 - BRIGADA MILITAR,FBR AVIATION INC,06 - SEGURANCA PUBLICA,25002232561,"R$ 170.789.344,00"
2,18 - ST,INDEFINIDO,Município Indefinido,Munícipio Indefinido,0110 - FUNRIGS PARCELAS DIVIDA,93 - INDENIZACOES E RESTITUICOES,9305 - INDENIZACOES,2321 - CONCESSAO PATROCINADA,2321.01.002 - REEQUILIBRIO CONTRATO CONCESSAO-RSM,24/4/2026,0049 - INDENIZACOES E RESTITUICOES,1801 - GABINETE E ORGAOS CENTRAI,CONCESSIONARIA ROTA DE SANTA MARIA S.A.,26 - TRANSPORTE,26002461919,"R$ 110.973.340,80"
3,16 - SEC. DESENV ECON,A CLASSIFICAR,A Classificar,Munícipio Indefinido,0108 - FUNRIGS - OUTRAS TRANSF,45 - SUBVENCOES ECONOMICAS,4504 - SUBSIDIO PARA QUITACAO DE DIVIDAS,3046 - CONC EMPR JURO ZERO,3046.01.002 - APOIO A CONSTITUIÇÃO DO PRONAMPE...,23/7/2024,0052 - SUBVENCOES,1601 - GABINETE E ORGAOS CENTRAI,BANCO DO ESTADO DO RIO GRANDE DO SUL S/A,23 - COMERCIO E SERVICOS,24004137877,"R$ 100.000.000,00"
4,17 - SEC HABITACAO,INDEFINIDO,Município Indefinido,Munícipio Indefinido,0110 - FUNRIGS PARCELAS DIVIDA,48 - OUTROS AUXILIOS FINANCEIROS A PESSOAS FIS...,4813 - SUBSIDIOS PROGRAMA PORTA DE ENTRADA,5415 - PROD ACOES HABITACIONAIS,5415.01.006 - PORTA DE ENTRADA,27/11/2024,0043 - AUXILIOS NAO SUJEITOS A COMPROVACAO,1783 - FEHIS,CAIXA ECONOMICA FEDERAL,16 - HABITACAO,24006572477,"R$ 100.000.000,00"


# Verificação das colunas

In [ ]:
print("Colunas disponíveis:\n")

print(
    repasse_municipio.columns.tolist()
)

Colunas disponíveis:

['Órgão', 'Região COREDE', 'Municipio', 'Situação Municipio', 'Recurso', 'Elemento', 'Rubrica', 'Projeto', 'Subprojeto', 'Data', 'Fato Contábil', 'Unidade Orçamentária', 'Credor', 'Função', 'Número Empenho', 'Pago/Repassado']


# Validação das colunas obrigatórias

In [ ]:
colunas_obrigatorias_repasse = [
    "Órgão",
    "Região COREDE",
    "Municipio",
    "Situação Municipio",
    "Recurso",
    "Elemento",
    "Rubrica",
    "Projeto",
    "Subprojeto",
    "Data",
    "Fato Contábil",
    "Unidade Orçamentária",
    "Credor",
    "Função",
    "Número Empenho",
    "Pago/Repassado"
]

colunas_ausentes_repasse = [
    coluna
    for coluna in colunas_obrigatorias_repasse
    if coluna not in repasse_municipio.columns
]

if colunas_ausentes_repasse:
    raise ValueError(
        "Colunas ausentes em REPASSE_MUNICIPIO.csv: "
        + ", ".join(colunas_ausentes_repasse)
    )

print(
    "✅ Todas as colunas obrigatórias foram identificadas."
)

✅ Todas as colunas obrigatórias foram identificadas.


# Renomeação das colunas

In [ ]:
repasse_municipio = (
    repasse_municipio
    .rename(
        columns={
            "Órgão": "Orgao_Repasse",
            "Região COREDE": "Regiao_COREDE",
            "Municipio": "Municipio_Repasse",
            "Situação Municipio": "Situacao_Municipio",
            "Recurso": "Recurso_Repasse",
            "Elemento": "Elemento_Repasse",
            "Rubrica": "Rubrica_Repasse",
            "Projeto": "Projeto_Repasse",
            "Subprojeto": "Subprojeto_Repasse",
            "Data": "Data_Repasse",
            "Fato Contábil": "Fato_Contabil",
            "Unidade Orçamentária": "Unidade_Orcamentaria_Repasse",
            "Credor": "Credor_Repasse",
            "Função": "Funcao_Repasse",
            "Número Empenho": "Numero_Empenho",
            "Pago/Repassado": "Valor_Pago_Repassado"
        }
    )
)

# Verificação após a renomeação

In [ ]:
print("Colunas após a renomeação:\n")

print(
    repasse_municipio.columns.tolist()
)

display(
    repasse_municipio.head()
)

Colunas após a renomeação:

['Orgao_Repasse', 'Regiao_COREDE', 'Municipio_Repasse', 'Situacao_Municipio', 'Recurso_Repasse', 'Elemento_Repasse', 'Rubrica_Repasse', 'Projeto_Repasse', 'Subprojeto_Repasse', 'Data_Repasse', 'Fato_Contabil', 'Unidade_Orcamentaria_Repasse', 'Credor_Repasse', 'Funcao_Repasse', 'Numero_Empenho', 'Valor_Pago_Repassado']


,Orgao_Repasse,Regiao_COREDE,Municipio_Repasse,Situacao_Municipio,Recurso_Repasse,Elemento_Repasse,Rubrica_Repasse,Projeto_Repasse,Subprojeto_Repasse,Data_Repasse,Fato_Contabil,Unidade_Orcamentaria_Repasse,Credor_Repasse,Funcao_Repasse,Numero_Empenho,Valor_Pago_Repassado
0,18 - SEC LOG E TRANSPORTES,INDEFINIDO,Município Indefinido,Munícipio Indefinido,0110 - FUNRIGS PARCELAS DIVIDA,65 - CONSTITUICAO OU AUMENTO DE CAPITAL DE EMP...,6503 - PARTICIPACAO EM CONSTITUICAO OU AUMENTO...,8107 - AUM CAP VINC - PORTOS RS,8107.01.001 - AUMENTO DE CAPITAL - PORTOS RS,19/12/2024,0053 - PARTICIPACOES SOCIETARIAS EM EMPRESAS S...,1801 - GABINETE E ORGAOS CENTRAI,PORTOS RS,26 - TRANSPORTE,24007402766,"R$ 731.389.734,00"
1,12 - SEC. DA SEGURANCA PUBLICA,INDEFINIDO,Município Indefinido,Munícipio Indefinido,0110 - FUNRIGS PARCELAS DIVIDA,52 - EQUIPAMENTOS E MATERIAL PERMANENTE,5226 - AERONAVES E/OU EQUIPAMENTOS PARA AERONAVES,5876 - QUALIF INSTAL E SV BM,5876.01.016 - AQUISCAO DE AERONAVES - ENFRENTA...,28/5/2025,0040 - FORNECIMENTO DE BENS E/OU SERVICOS - NA...,1203 - BRIGADA MILITAR,FBR AVIATION INC,06 - SEGURANCA PUBLICA,25002232561,"R$ 170.789.344,00"
2,18 - ST,INDEFINIDO,Município Indefinido,Munícipio Indefinido,0110 - FUNRIGS PARCELAS DIVIDA,93 - INDENIZACOES E RESTITUICOES,9305 - INDENIZACOES,2321 - CONCESSAO PATROCINADA,2321.01.002 - REEQUILIBRIO CONTRATO CONCESSAO-RSM,24/4/2026,0049 - INDENIZACOES E RESTITUICOES,1801 - GABINETE E ORGAOS CENTRAI,CONCESSIONARIA ROTA DE SANTA MARIA S.A.,26 - TRANSPORTE,26002461919,"R$ 110.973.340,80"
3,16 - SEC. DESENV ECON,A CLASSIFICAR,A Classificar,Munícipio Indefinido,0108 - FUNRIGS - OUTRAS TRANSF,45 - SUBVENCOES ECONOMICAS,4504 - SUBSIDIO PARA QUITACAO DE DIVIDAS,3046 - CONC EMPR JURO ZERO,3046.01.002 - APOIO A CONSTITUIÇÃO DO PRONAMPE...,23/7/2024,0052 - SUBVENCOES,1601 - GABINETE E ORGAOS CENTRAI,BANCO DO ESTADO DO RIO GRANDE DO SUL S/A,23 - COMERCIO E SERVICOS,24004137877,"R$ 100.000.000,00"
4,17 - SEC HABITACAO,INDEFINIDO,Município Indefinido,Munícipio Indefinido,0110 - FUNRIGS PARCELAS DIVIDA,48 - OUTROS AUXILIOS FINANCEIROS A PESSOAS FIS...,4813 - SUBSIDIOS PROGRAMA PORTA DE ENTRADA,5415 - PROD ACOES HABITACIONAIS,5415.01.006 - PORTA DE ENTRADA,27/11/2024,0043 - AUXILIOS NAO SUJEITOS A COMPROVACAO,1783 - FEHIS,CAIXA ECONOMICA FEDERAL,16 - HABITACAO,24006572477,"R$ 100.000.000,00"


# Criação da chave técnica

In [ ]:
repasse_municipio = (
    repasse_municipio
    .reset_index(drop=True)
    .copy()
)

repasse_municipio.insert(
    0,
    "ID_Repasse_Municipio",
    (
        repasse_municipio.index
        + 1
    )
)

repasse_municipio[
    "ID_Repasse_Municipio"
] = (
    repasse_municipio[
        "ID_Repasse_Municipio"
    ]
    .astype("Int64")
)

print(
    "IDs únicos:",
    repasse_municipio[
        "ID_Repasse_Municipio"
    ].nunique()
)

print(
    "Registros totais:",
    len(repasse_municipio)
)

IDs únicos: 5868
Registros totais: 5868


# Tratamento do valor pago ou repassado

In [ ]:
repasse_municipio[
    "Valor_Pago_Repassado_Original"
] = (
    repasse_municipio[
        "Valor_Pago_Repassado"
    ]
    .copy()
)

repasse_municipio[
    "Valor_Pago_Repassado"
] = (
    repasse_municipio[
        "Valor_Pago_Repassado"
    ]
    .astype("string")
    .str.replace(
        "R$",
        "",
        regex=False
    )
    .str.replace(
        ".",
        "",
        regex=False
    )
    .str.replace(
        ",",
        ".",
        regex=False
    )
    .str.strip()
)

repasse_municipio[
    "Valor_Pago_Repassado"
] = pd.to_numeric(
    repasse_municipio[
        "Valor_Pago_Repassado"
    ],
    errors="coerce"
).round(2)

# Validação da conversão financeira

In [ ]:
print(
    "Valores nulos após a conversão:",
    repasse_municipio[
        "Valor_Pago_Repassado"
    ].isna().sum()
)

print(
    "Valor total pago/repassado: "
    f"R$ "
    f"{repasse_municipio['Valor_Pago_Repassado'].sum():,.2f}"
)

Valores nulos após a conversão: 0
Valor total pago/repassado: R$ 3,349,113,382.59


# Conversão da data

In [ ]:
repasse_municipio[
    "Data_Repasse"
] = pd.to_datetime(
    repasse_municipio[
        "Data_Repasse"
    ],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Validação das datas

In [ ]:
print(
    "Datas não convertidas:",
    repasse_municipio[
        "Data_Repasse"
    ].isna().sum()
)

print(
    "Data inicial:",
    repasse_municipio[
        "Data_Repasse"
    ].min()
)

print(
    "Data final:",
    repasse_municipio[
        "Data_Repasse"
    ].max()
)

Datas não convertidas: 0
Data inicial: 2024-05-14 00:00:00
Data final: 2026-09-15 00:00:00


# Padronização do número do empenho

In [ ]:
repasse_municipio[
    "Numero_Empenho"
] = (
    repasse_municipio[
        "Numero_Empenho"
    ]
    .astype("string")
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
    .str.replace(
        r"\D",
        "",
        regex=True
    )
    .replace(
        "",
        pd.NA
    )
)

# Validação dos empenhos

In [ ]:
print(
    "Registros com número de empenho:",
    repasse_municipio[
        "Numero_Empenho"
    ].notna().sum()
)

print(
    "Registros sem número de empenho:",
    repasse_municipio[
        "Numero_Empenho"
    ].isna().sum()
)

print(
    "Números de empenho únicos:",
    repasse_municipio[
        "Numero_Empenho"
    ].nunique()
)

Registros com número de empenho: 5868
Registros sem número de empenho: 0
Números de empenho únicos: 5011


# Padronização dos campos textuais

In [ ]:
colunas_texto_repasse = [
    "Orgao_Repasse",
    "Regiao_COREDE",
    "Municipio_Repasse",
    "Situacao_Municipio",
    "Recurso_Repasse",
    "Elemento_Repasse",
    "Rubrica_Repasse",
    "Projeto_Repasse",
    "Subprojeto_Repasse",
    "Fato_Contabil",
    "Unidade_Orcamentaria_Repasse",
    "Credor_Repasse",
    "Funcao_Repasse"
]

for coluna in colunas_texto_repasse:

    repasse_municipio[coluna] = (
        repasse_municipio[coluna]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

# Classificação da identificação municipal

In [ ]:
mascara_municipio_indefinido = (
    repasse_municipio[
        "Municipio_Repasse"
    ].isna()
    |
    repasse_municipio[
        "Municipio_Repasse"
    ]
    .astype("string")
    .str.contains(
        r"indefinido|a classificar",
        case=False,
        na=False,
        regex=True
    )
)

repasse_municipio[
    "Status_Identificacao_Municipio"
] = np.where(
    mascara_municipio_indefinido,
    "Município não identificado",
    "Município identificado"
)

repasse_municipio[
    "Ind_Municipio_Identificado"
] = np.where(
    mascara_municipio_indefinido,
    0,
    1
).astype("int8")

# Validação da identificação municipal

In [ ]:
resumo_identificacao_municipio = (
    repasse_municipio
    .groupby(
        "Status_Identificacao_Municipio",
        as_index=False
    )
    .agg(
        Quantidade_Registros=(
            "ID_Repasse_Municipio",
            "count"
        ),
        Valor_Total=(
            "Valor_Pago_Repassado",
            "sum"
        )
    )
)

resumo_identificacao_municipio[
    "Percentual_Registros"
] = (
    resumo_identificacao_municipio[
        "Quantidade_Registros"
    ]
    .div(len(repasse_municipio))
    .mul(100)
    .round(2)
)

resumo_identificacao_municipio[
    "Percentual_Valor"
] = (
    resumo_identificacao_municipio[
        "Valor_Total"
    ]
    .div(
        repasse_municipio[
            "Valor_Pago_Repassado"
        ].sum()
    )
    .mul(100)
    .round(2)
)

display(
    resumo_identificacao_municipio
)

,Status_Identificacao_Municipio,Quantidade_Registros,Valor_Total,Percentual_Registros,Percentual_Valor
0,Município identificado,4098,687753939.15,69.84,20.54
1,Município não identificado,1770,2661359443.44,30.16,79.46


# Resultados da identificação municipal

In [ ]:
quantidade_identificados = (
    repasse_municipio[
        "Ind_Municipio_Identificado"
    ]
    .eq(1)
    .sum()
)

quantidade_indefinidos = (
    repasse_municipio[
        "Ind_Municipio_Identificado"
    ]
    .eq(0)
    .sum()
)

valor_identificado = (
    repasse_municipio.loc[
        repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1),
        "Valor_Pago_Repassado"
    ]
    .sum()
)

valor_indefinido = (
    repasse_municipio.loc[
        repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(0),
        "Valor_Pago_Repassado"
    ]
    .sum()
)

print(
    "Registros com município identificado:",
    quantidade_identificados
)

print(
    "Registros com município não identificado:",
    quantidade_indefinidos
)

print(
    "Valor com município identificado: "
    f"R$ {valor_identificado:,.2f}"
)

print(
    "Valor com município não identificado: "
    f"R$ {valor_indefinido:,.2f}"
)

Registros com município identificado: 4098
Registros com município não identificado: 1770
Valor com município identificado: R$ 687,753,939.15
Valor com município não identificado: R$ 2,661,359,443.44


# Normalização do nome do município

In [ ]:
import re
def normalizar_nome_municipio(valor):

    if pd.isna(valor):
        return pd.NA

    valor = str(valor).strip()

    if valor == "":
        return pd.NA

    valor = unidecode(valor)

    valor = valor.upper()

    valor = re.sub(
        r"[^A-Z0-9 ]",
        " ",
        valor
    )

    valor = re.sub(
        r"\s+",
        " ",
        valor
    ).strip()

    return valor

# Teste da função

In [ ]:
municipios_teste = [
    "São Leopoldo",
    "Caxias do Sul",
    "São Sebastião do Caí",
    "  Porto Alegre  ",
    pd.NA
]

for municipio_teste in municipios_teste:

    print(
        municipio_teste,
        "=>",
        normalizar_nome_municipio(
            municipio_teste
        )
    )

São Leopoldo => SAO LEOPOLDO
Caxias do Sul => CAXIAS DO SUL
São Sebastião do Caí => SAO SEBASTIAO DO CAI
  Porto Alegre   => PORTO ALEGRE
<NA> => <NA>


# Criação das chaves municipais normalizadas

In [ ]:
repasse_municipio[
    "Municipio_Repasse_Key"
] = (
    repasse_municipio[
        "Municipio_Repasse"
    ]
    .apply(
        normalizar_nome_municipio
    )
    .astype("string")
)

dim_municipio[
    "Municipio_Key"
] = (
    dim_municipio[
        "Municipio"
    ]
    .apply(
        normalizar_nome_municipio
    )
    .astype("string")
)

# Verificação das chaves criadas

In [ ]:
display(
    repasse_municipio[
        [
            "Municipio_Repasse",
            "Municipio_Repasse_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Municipio_Repasse"
    )
    .head(30)
)

display(
    dim_municipio[
        [
            "Municipio",
            "Municipio_Key"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Municipio"
    )
    .head(30)
)

,Municipio_Repasse,Municipio_Repasse_Key
3,A Classificar,A CLASSIFICAR
2182,Aceguá,ACEGUA
180,Agudo,AGUDO
1606,Ajuricaba,AJURICABA
1536,Alecrim,ALECRIM
1545,Alegrete,ALEGRETE
2872,Alegria,ALEGRIA
2219,Alpestre,ALPESTRE
1482,Alto Alegre,ALTO ALEGRE
1726,Alto Feliz,ALTO FELIZ


,Municipio,Municipio_Key
0,Aceguá,ACEGUA
2,Agudo,AGUDO
3,Ajuricaba,AJURICABA
4,Alecrim,ALECRIM
5,Alegrete,ALEGRETE
6,Alegria,ALEGRIA
7,Almirante Tamandaré do Sul,ALMIRANTE TAMANDARE DO SUL
8,Alpestre,ALPESTRE
9,Alto Alegre,ALTO ALEGRE
10,Alto Feliz,ALTO FELIZ


# Validação das chaves

In [ ]:
print(
    "Chaves nulas em REPASSE_MUNICIPIO:",
    repasse_municipio[
        "Municipio_Repasse_Key"
    ].isna().sum()
)

print(
    "Chaves nulas em DIM_MUNICIPIO:",
    dim_municipio[
        "Municipio_Key"
    ].isna().sum()
)

print(
    "Municípios distintos na fonte:",
    repasse_municipio.loc[
        repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1),
        "Municipio_Repasse_Key"
    ].nunique()
)

Chaves nulas em REPASSE_MUNICIPIO: 0
Chaves nulas em DIM_MUNICIPIO: 0
Municípios distintos na fonte: 409


# Preparação do mapa municipal

In [ ]:
mapa_codigo_municipio = (
    dim_municipio[
        [
            "Cod_IBGE",
            "Municipio",
            "Municipio_Key"
        ]
    ]
    .drop_duplicates(
        subset=["Municipio_Key"]
    )
    .rename(
        columns={
            "Municipio":
            "Municipio_DIM"
        }
    )
    .copy()
)

# Validação da unicidade da chave municipal

In [ ]:
duplicidades_municipio_key = (
    mapa_codigo_municipio[
        "Municipio_Key"
    ]
    .duplicated()
    .sum()
)

print(
    "Chaves municipais duplicadas:",
    duplicidades_municipio_key
)

if duplicidades_municipio_key > 0:
    raise ValueError(
        "A DIM_MUNICIPIO possui chaves "
        "municipais normalizadas duplicadas."
    )

Chaves municipais duplicadas: 0


# Inclusão do Código IBGE

In [ ]:
fact_repasse_municipio = (
    repasse_municipio
    .merge(
        mapa_codigo_municipio,
        left_on="Municipio_Repasse_Key",
        right_on="Municipio_Key",
        how="left",
        validate="many_to_one"
    )
)

# Remoção da chave duplicada da dimensão

In [ ]:
fact_repasse_municipio = (
    fact_repasse_municipio
    .drop(
        columns=[
            "Municipio_Key"
        ],
        errors="ignore"
    )
)

# Validação da integração municipal

In [ ]:
municipios_identificados = (
    fact_repasse_municipio[
        fact_repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1)
    ]
    .copy()
)

municipios_sem_codigo = (
    municipios_identificados[
        municipios_identificados[
            "Cod_IBGE"
        ].isna()
    ]
)

print(
    "Registros territorializados:",
    len(municipios_identificados)
)

print(
    "Registros territorializados sem Código IBGE:",
    len(municipios_sem_codigo)
)

print(
    "Municípios informados na fonte:",
    municipios_identificados[
        "Municipio_Repasse"
    ].nunique()
)

print(
    "Municípios com Código IBGE:",
    municipios_identificados.loc[
        municipios_identificados[
            "Cod_IBGE"
        ].notna(),
        "Cod_IBGE"
    ].nunique()
)

Registros territorializados: 4098
Registros territorializados sem Código IBGE: 2
Municípios informados na fonte: 409
Municípios com Código IBGE: 408


# Inspeção das diferenças municipais

In [ ]:
if len(municipios_sem_codigo) > 0:

    municipios_pendentes = (
        municipios_sem_codigo[
            [
                "Municipio_Repasse",
                "Municipio_Repasse_Key",
                "Regiao_COREDE",
                "Situacao_Municipio"
            ]
        ]
        .drop_duplicates()
        .sort_values(
            "Municipio_Repasse"
        )
    )

    print(
        "Municípios distintos sem Código IBGE:",
        municipios_pendentes[
            "Municipio_Repasse"
        ].nunique()
    )

    display(
        municipios_pendentes
    )

else:

    print(
        "✅ Todos os municípios identificados "
        "possuem Código IBGE."
    )

Municípios distintos sem Código IBGE: 1


,Municipio_Repasse,Municipio_Repasse_Key,Regiao_COREDE,Situacao_Municipio
1816,Santana do Livramento,SANTANA DO LIVRAMENTO,FRONTEIRA OESTE,Emergência


# Garantia de nulo para municípios indefinidos

In [ ]:
fact_repasse_municipio.loc[
    fact_repasse_municipio[
        "Ind_Municipio_Identificado"
    ].eq(0),
    "Cod_IBGE"
] = pd.NA

fact_repasse_municipio.loc[
    fact_repasse_municipio[
        "Ind_Municipio_Identificado"
    ].eq(0),
    "Municipio_DIM"
] = pd.NA

# Conversão do Código IBGE

In [ ]:
fact_repasse_municipio[
    "Cod_IBGE"
] = pd.to_numeric(
    fact_repasse_municipio[
        "Cod_IBGE"
    ],
    errors="coerce"
).astype("Int64")

# Validação da quantidade de registros

In [ ]:
print(
    "Registros na base original:",
    len(repasse_municipio)
)

print(
    "Registros na FACT_REPASSE_MUNICIPIO:",
    len(fact_repasse_municipio)
)

diferenca_registros = (
    len(fact_repasse_municipio)
    - len(repasse_municipio)
)

print(
    "Diferença de registros:",
    diferenca_registros
)

if diferenca_registros != 0:
    raise ValueError(
        "A integração municipal alterou "
        "a quantidade de registros."
    )

Registros na base original: 5868
Registros na FACT_REPASSE_MUNICIPIO: 5868
Diferença de registros: 0


# Validação da preservação financeira

In [ ]:
valor_antes_integracao = round(
    repasse_municipio[
        "Valor_Pago_Repassado"
    ].sum(),
    2
)

valor_depois_integracao = round(
    fact_repasse_municipio[
        "Valor_Pago_Repassado"
    ].sum(),
    2
)

diferenca_financeira_repasse = round(
    valor_antes_integracao
    - valor_depois_integracao,
    2
)

print(
    "Valor antes da integração: "
    f"R$ {valor_antes_integracao:,.2f}"
)

print(
    "Valor depois da integração: "
    f"R$ {valor_depois_integracao:,.2f}"
)

print(
    "Diferença financeira:",
    diferenca_financeira_repasse
)

if diferenca_financeira_repasse != 0:
    raise ValueError(
        "A integração municipal alterou "
        "o valor financeiro total."
    )

Valor antes da integração: R$ 3,349,113,382.59
Valor depois da integração: R$ 3,349,113,382.59
Diferença financeira: 0.0


# Validação de registros duplicados

In [ ]:
colunas_registro_original = [
    "Orgao_Repasse",
    "Regiao_COREDE",
    "Municipio_Repasse",
    "Situacao_Municipio",
    "Recurso_Repasse",
    "Elemento_Repasse",
    "Rubrica_Repasse",
    "Projeto_Repasse",
    "Subprojeto_Repasse",
    "Data_Repasse",
    "Fato_Contabil",
    "Unidade_Orcamentaria_Repasse",
    "Credor_Repasse",
    "Funcao_Repasse",
    "Numero_Empenho",
    "Valor_Pago_Repassado"
]

duplicados_exatos_repasse = (
    fact_repasse_municipio[
        colunas_registro_original
    ]
    .duplicated()
    .sum()
)

print(
    "Registros integralmente duplicados:",
    duplicados_exatos_repasse
)

Registros integralmente duplicados: 0


### Se existirem duplicados exatos, eles deverão ser inspecionados antes de qualquer exclusão:

In [ ]:
if duplicados_exatos_repasse > 0:

    display(
        fact_repasse_municipio[
            fact_repasse_municipio[
                colunas_registro_original
            ]
            .duplicated(
                keep=False
            )
        ]
        .sort_values(
            [
                "Numero_Empenho",
                "Data_Repasse"
            ]
        )
    )

# Distribuição por situação municipal

In [ ]:
repasse_por_situacao = (
    fact_repasse_municipio
    .groupby(
        "Situacao_Municipio",
        dropna=False,
        as_index=False
    )
    .agg(
        Quantidade_Registros=(
            "ID_Repasse_Municipio",
            "count"
        ),
        Valor_Total=(
            "Valor_Pago_Repassado",
            "sum"
        )
    )
    .sort_values(
        "Valor_Total",
        ascending=False
    )
)

repasse_por_situacao[
    "Valor_Total"
] = (
    repasse_por_situacao[
        "Valor_Total"
    ]
    .round(2)
)

display(
    repasse_por_situacao
)

,Situacao_Municipio,Quantidade_Registros,Valor_Total
2,Munícipio Indefinido,1770,2661359443.44
0,Calamidade,2718,572126564.22
1,Emergência,1311,112519374.93
3,Não Homologado,69,3108000.0


# Distribuição por Região COREDE

In [ ]:
repasse_por_corede = (
    fact_repasse_municipio[
        fact_repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1)
    ]
    .groupby(
        "Regiao_COREDE",
        as_index=False
    )
    .agg(
        Quantidade_Registros=(
            "ID_Repasse_Municipio",
            "count"
        ),
        Quantidade_Municipios=(
            "Cod_IBGE",
            "nunique"
        ),
        Valor_Total=(
            "Valor_Pago_Repassado",
            "sum"
        )
    )
    .sort_values(
        "Valor_Total",
        ascending=False
    )
)

repasse_por_corede[
    "Valor_Total"
] = (
    repasse_por_corede[
        "Valor_Total"
    ]
    .round(2)
)

display(
    repasse_por_corede.head(20)
)

,Regiao_COREDE,Quantidade_Registros,Quantidade_Municipios,Valor_Total
13,METROPOLITANO DELTA DO JACUI,1131,10,226746633.32
27,VALE DO TAQUARI,305,34,104030949.39
25,VALE DO RIO DOS SINOS,449,14,68632577.2
26,VALE DO RIO PARDO,290,22,44666032.63
21,SERRA,222,28,37000448.14
5,CENTRAL,208,17,30153170.53
22,SUL,186,19,27220952.13
23,VALE DO CAI,148,19,24403431.57
18,PARANHANA-ENCOSTA SERRA,100,9,18781881.53
6,CENTRO SUL,165,15,15401108.0


# Principais municípios por valor pago ou repassado

In [ ]:
repasse_por_municipio = (
    fact_repasse_municipio[
        fact_repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1)
    ]
    .dropna(
        subset=["Cod_IBGE"]
    )
    .groupby(
        [
            "Cod_IBGE",
            "Municipio_Repasse",
            "Regiao_COREDE",
            "Situacao_Municipio"
        ],
        as_index=False
    )
    .agg(
        Quantidade_Registros=(
            "ID_Repasse_Municipio",
            "count"
        ),
        Quantidade_Empenhos=(
            "Numero_Empenho",
            "nunique"
        ),
        Valor_Total=(
            "Valor_Pago_Repassado",
            "sum"
        )
    )
    .sort_values(
        "Valor_Total",
        ascending=False
    )
)

repasse_por_municipio[
    "Valor_Total"
] = (
    repasse_por_municipio[
        "Valor_Total"
    ]
    .round(2)
)

display(
    repasse_por_municipio.head(20)
)

,Cod_IBGE,Municipio_Repasse,Regiao_COREDE,Situacao_Municipio,Quantidade_Registros,Quantidade_Empenhos,Valor_Total
265,4314902,Porto Alegre,METROPOLITANO DELTA DO JACUI,Calamidade,851,749,127398545.11
115,4306767,Eldorado do Sul,METROPOLITANO DELTA DO JACUI,Calamidade,39,34,67833097.6
63,4304606,Canoas,VALE DO RIO DOS SINOS,Calamidade,164,163,24060537.37
326,4318705,São Leopoldo,VALE DO RIO DOS SINOS,Calamidade,90,88,17742958.14
128,4307807,Estrela,VALE DO TAQUARI,Calamidade,34,32,14694815.34
101,4306205,Cruzeiro do Sul,VALE DO TAQUARI,Calamidade,27,24,14080122.17
116,4306809,Encantado,VALE DO TAQUARI,Calamidade,26,21,14047249.68
381,4322004,Triunfo,METROPOLITANO DELTA DO JACUI,Calamidade,30,29,9785111.54
17,4301008,Arroio do Meio,VALE DO TAQUARI,Calamidade,22,21,9451043.05
283,4315800,Roca Sales,VALE DO TAQUARI,Calamidade,15,14,8643887.37


# Indicadores de cobertura territorial

In [ ]:
valor_total_repasse = (
    fact_repasse_municipio[
        "Valor_Pago_Repassado"
    ]
    .sum()
)

valor_com_municipio = (
    fact_repasse_municipio.loc[
        fact_repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1),
        "Valor_Pago_Repassado"
    ]
    .sum()
)

valor_sem_municipio = (
    fact_repasse_municipio.loc[
        fact_repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(0),
        "Valor_Pago_Repassado"
    ]
    .sum()
)

percentual_valor_territorializado = round(
    valor_com_municipio
    / valor_total_repasse
    * 100,
    2
)

percentual_valor_indefinido = round(
    valor_sem_municipio
    / valor_total_repasse
    * 100,
    2
)

print(
    "Valor total da base: "
    f"R$ {valor_total_repasse:,.2f}"
)

print(
    "Valor com município identificado: "
    f"R$ {valor_com_municipio:,.2f}"
)

print(
    "Valor com município não identificado: "
    f"R$ {valor_sem_municipio:,.2f}"
)

print(
    "Percentual do valor territorializado:",
    f"{percentual_valor_territorializado}%"
)

print(
    "Percentual do valor não territorializado:",
    f"{percentual_valor_indefinido}%"
)

Valor total da base: R$ 3,349,113,382.59
Valor com município identificado: R$ 687,753,939.15
Valor com município não identificado: R$ 2,661,359,443.44
Percentual do valor territorializado: 20.54%
Percentual do valor não territorializado: 79.46%


# Estrutura final da FACT_REPASSE_MUNICIPIO

In [ ]:
colunas_fact_repasse = [
    "ID_Repasse_Municipio",
    "Cod_IBGE",
    "Municipio_Repasse",
    "Regiao_COREDE",
    "Situacao_Municipio",
    "Status_Identificacao_Municipio",
    "Ind_Municipio_Identificado",
    "Data_Repasse",
    "Numero_Empenho",
    "Valor_Pago_Repassado",
    "Orgao_Repasse",
    "Unidade_Orcamentaria_Repasse",
    "Credor_Repasse",
    "Recurso_Repasse",
    "Elemento_Repasse",
    "Rubrica_Repasse",
    "Projeto_Repasse",
    "Subprojeto_Repasse",
    "Fato_Contabil",
    "Funcao_Repasse"
]

fact_repasse_municipio = (
    fact_repasse_municipio[
        colunas_fact_repasse
    ]
    .copy()
)

# Verificação da estrutura final

In [ ]:
print("FACT_REPASSE_MUNICIPIO")
print("-" * 50)

print(
    "Quantidade final de registros:",
    len(fact_repasse_municipio)
)

print(
    "Quantidade final de colunas:",
    fact_repasse_municipio.shape[1]
)

print(
    "Municípios identificados:",
    fact_repasse_municipio.loc[
        fact_repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1),
        "Cod_IBGE"
    ].nunique()
)

print(
    "Valor total: "
    f"R$ "
    f"{fact_repasse_municipio['Valor_Pago_Repassado'].sum():,.2f}"
)

display(
    fact_repasse_municipio.head()
)

FACT_REPASSE_MUNICIPIO
--------------------------------------------------
Quantidade final de registros: 5868
Quantidade final de colunas: 20
Municípios identificados: 408
Valor total: R$ 3,349,113,382.59


,ID_Repasse_Municipio,Cod_IBGE,Municipio_Repasse,Regiao_COREDE,Situacao_Municipio,Status_Identificacao_Municipio,Ind_Municipio_Identificado,Data_Repasse,Numero_Empenho,Valor_Pago_Repassado,Orgao_Repasse,Unidade_Orcamentaria_Repasse,Credor_Repasse,Recurso_Repasse,Elemento_Repasse,Rubrica_Repasse,Projeto_Repasse,Subprojeto_Repasse,Fato_Contabil,Funcao_Repasse
0,1,<NA>,Município Indefinido,INDEFINIDO,Munícipio Indefinido,Município não identificado,0,2024-12-19,24007402766,731389734.0,18 - SEC LOG E TRANSPORTES,1801 - GABINETE E ORGAOS CENTRAI,PORTOS RS,0110 - FUNRIGS PARCELAS DIVIDA,65 - CONSTITUICAO OU AUMENTO DE CAPITAL DE EMP...,6503 - PARTICIPACAO EM CONSTITUICAO OU AUMENTO...,8107 - AUM CAP VINC - PORTOS RS,8107.01.001 - AUMENTO DE CAPITAL - PORTOS RS,0053 - PARTICIPACOES SOCIETARIAS EM EMPRESAS S...,26 - TRANSPORTE
1,2,<NA>,Município Indefinido,INDEFINIDO,Munícipio Indefinido,Município não identificado,0,2025-05-28,25002232561,170789344.0,12 - SEC. DA SEGURANCA PUBLICA,1203 - BRIGADA MILITAR,FBR AVIATION INC,0110 - FUNRIGS PARCELAS DIVIDA,52 - EQUIPAMENTOS E MATERIAL PERMANENTE,5226 - AERONAVES E/OU EQUIPAMENTOS PARA AERONAVES,5876 - QUALIF INSTAL E SV BM,5876.01.016 - AQUISCAO DE AERONAVES - ENFRENTA...,0040 - FORNECIMENTO DE BENS E/OU SERVICOS - NA...,06 - SEGURANCA PUBLICA
2,3,<NA>,Município Indefinido,INDEFINIDO,Munícipio Indefinido,Município não identificado,0,2026-04-24,26002461919,110973340.8,18 - ST,1801 - GABINETE E ORGAOS CENTRAI,CONCESSIONARIA ROTA DE SANTA MARIA S.A.,0110 - FUNRIGS PARCELAS DIVIDA,93 - INDENIZACOES E RESTITUICOES,9305 - INDENIZACOES,2321 - CONCESSAO PATROCINADA,2321.01.002 - REEQUILIBRIO CONTRATO CONCESSAO-RSM,0049 - INDENIZACOES E RESTITUICOES,26 - TRANSPORTE
3,4,<NA>,A Classificar,A CLASSIFICAR,Munícipio Indefinido,Município não identificado,0,2024-07-23,24004137877,100000000.0,16 - SEC. DESENV ECON,1601 - GABINETE E ORGAOS CENTRAI,BANCO DO ESTADO DO RIO GRANDE DO SUL S/A,0108 - FUNRIGS - OUTRAS TRANSF,45 - SUBVENCOES ECONOMICAS,4504 - SUBSIDIO PARA QUITACAO DE DIVIDAS,3046 - CONC EMPR JURO ZERO,3046.01.002 - APOIO A CONSTITUIÇÃO DO PRONAMPE...,0052 - SUBVENCOES,23 - COMERCIO E SERVICOS
4,5,<NA>,Município Indefinido,INDEFINIDO,Munícipio Indefinido,Município não identificado,0,2024-11-27,24006572477,100000000.0,17 - SEC HABITACAO,1783 - FEHIS,CAIXA ECONOMICA FEDERAL,0110 - FUNRIGS PARCELAS DIVIDA,48 - OUTROS AUXILIOS FINANCEIROS A PESSOAS FIS...,4813 - SUBSIDIOS PROGRAMA PORTA DE ENTRADA,5415 - PROD ACOES HABITACIONAIS,5415.01.006 - PORTA DE ENTRADA,0043 - AUXILIOS NAO SUJEITOS A COMPROVACAO,16 - HABITACAO


# Validação final

In [ ]:
print(
    "Validação Final da FACT_REPASSE_MUNICIPIO"
)

print("-" * 50)

print(
    "Registros:",
    len(fact_repasse_municipio)
)

print(
    "IDs únicos:",
    fact_repasse_municipio[
        "ID_Repasse_Municipio"
    ].nunique()
)

print(
    "Empenhos únicos:",
    fact_repasse_municipio[
        "Numero_Empenho"
    ].nunique()
)

print(
    "Registros com município identificado:",
    fact_repasse_municipio[
        "Ind_Municipio_Identificado"
    ].eq(1).sum()
)

print(
    "Registros sem município identificado:",
    fact_repasse_municipio[
        "Ind_Municipio_Identificado"
    ].eq(0).sum()
)

print(
    "Registros territorializados sem Código IBGE:",
    fact_repasse_municipio[
        fact_repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1)
    ][
        "Cod_IBGE"
    ].isna().sum()
)

print(
    "Diferença financeira:",
    diferenca_financeira_repasse
)

Validação Final da FACT_REPASSE_MUNICIPIO
--------------------------------------------------
Registros: 5868
IDs únicos: 5868
Empenhos únicos: 5011
Registros com município identificado: 4098
Registros sem município identificado: 1770
Registros territorializados sem Código IBGE: 2
Diferença financeira: 0.0


# Exportação da FACT_REPASSE_MUNICIPIO

In [ ]:
arquivo_saida_fact_repasse = os.path.join(
    pasta_clean,
    "FACT_REPASSE_MUNICIPIO.xlsx"
)

fact_repasse_municipio.to_excel(
    arquivo_saida_fact_repasse,
    index=False
)

print(
    "✅ FACT_REPASSE_MUNICIPIO.xlsx "
    "criado com sucesso."
)

✅ FACT_REPASSE_MUNICIPIO.xlsx criado com sucesso.


# Validação da exportação

In [ ]:
if os.path.exists(
    arquivo_saida_fact_repasse
):

    tamanho_mb = (
        os.path.getsize(
            arquivo_saida_fact_repasse
        )
        / 1024
        / 1024
    )

    print(
        "✅ Arquivo exportado: "
        "FACT_REPASSE_MUNICIPIO.xlsx"
    )

    print(
        f"Tamanho do arquivo: "
        f"{tamanho_mb:.2f} MB"
    )

else:

    print(
        "❌ O arquivo "
        "FACT_REPASSE_MUNICIPIO.xlsx "
        "não foi encontrado."
    )

✅ Arquivo exportado: FACT_REPASSE_MUNICIPIO.xlsx
Tamanho do arquivo: 0.64 MB


# Resumo Final

In [ ]:
resumo_modelo = pd.DataFrame({
    "Tabela": [
        "DIM_MUNICIPIO",
        "FACT_IMPACTO_MUNICIPIO",
        "FACT_REPASSE_MUNICIPIO",
        "FACT_PAGAMENTOS",
        "DIM_CREDOR_CNPJ",
        "DIM_INDICADORES_SOCIAIS_RS"
    ],
    "Linhas": [
        len(dim_municipio),
        len(fact_impacto),
        len(fact_repasse_municipio),
        len(fact_pagamentos),
        len(dim_credor_cnpj),
        len(df_vuln)
    ],
    "Colunas": [
        dim_municipio.shape[1],
        fact_impacto.shape[1],
        fact_repasse_municipio.shape[1],
        fact_pagamentos.shape[1],
        dim_credor_cnpj.shape[1],
        df_vuln.shape[1]
    ]
})

display(
    resumo_modelo
)

,Tabela,Linhas,Colunas
0,DIM_MUNICIPIO,497,11
1,FACT_IMPACTO_MUNICIPIO,68,9
2,FACT_REPASSE_MUNICIPIO,5868,20
3,FACT_PAGAMENTOS,38210,23
4,DIM_CREDOR_CNPJ,8023,21
5,DIM_INDICADORES_SOCIAIS_RS,1,20


# Verificação final de nomes compartilhados

In [ ]:
tabelas_modelo = {
    "DIM_MUNICIPIO":
        dim_municipio,

    "FACT_IMPACTO_MUNICIPIO":
        fact_impacto,

    "FACT_REPASSE_MUNICIPIO":
        fact_repasse_municipio,

    "FACT_PAGAMENTOS":
        fact_pagamentos,

    "DIM_CREDOR_CNPJ":
        dim_credor_cnpj,

    "DIM_INDICADORES_SOCIAIS_RS":
        df_vuln
}

nomes_tabelas = list(
    tabelas_modelo.keys()
)

for indice_a in range(
    len(nomes_tabelas)
):

    for indice_b in range(
        indice_a + 1,
        len(nomes_tabelas)
    ):

        tabela_a = nomes_tabelas[
            indice_a
        ]

        tabela_b = nomes_tabelas[
            indice_b
        ]

        campos_comuns = sorted(
            set(
                tabelas_modelo[
                    tabela_a
                ].columns
            )
            &
            set(
                tabelas_modelo[
                    tabela_b
                ].columns
            )
        )

        if campos_comuns:

            print(
                f"{tabela_a} "
                f"<-> "
                f"{tabela_b}: "
                f"{campos_comuns}"
            )

DIM_MUNICIPIO <-> FACT_IMPACTO_MUNICIPIO: ['Cod_IBGE']
DIM_MUNICIPIO <-> FACT_REPASSE_MUNICIPIO: ['Cod_IBGE']
DIM_MUNICIPIO <-> DIM_INDICADORES_SOCIAIS_RS: ['IDHM']
FACT_IMPACTO_MUNICIPIO <-> FACT_REPASSE_MUNICIPIO: ['Cod_IBGE']
FACT_PAGAMENTOS <-> DIM_CREDOR_CNPJ: ['Documento_Original']


# Validação final do modelo

In [ ]:
print("VALIDAÇÃO FINAL DO MODELO")
print("-" * 60)

print(
    "DIM_MUNICIPIO:",
    len(dim_municipio)
)

print(
    "FACT_IMPACTO_MUNICIPIO:",
    len(fact_impacto)
)

print(
    "FACT_REPASSE_MUNICIPIO:",
    len(fact_repasse_municipio)
)

print(
    "FACT_PAGAMENTOS:",
    len(fact_pagamentos)
)

print(
    "DIM_CREDOR_CNPJ:",
    len(dim_credor_cnpj)
)

print(
    "DIM_INDICADORES_SOCIAIS_RS:",
    len(df_vuln)
)

print("-" * 60)

print(
    "Impactos sem Código IBGE:",
    fact_impacto[
        "Cod_IBGE"
    ].isna().sum()
)

print(
    "Repasses territorializados sem Código IBGE:",
    fact_repasse_municipio.loc[
        fact_repasse_municipio[
            "Ind_Municipio_Identificado"
        ].eq(1),
        "Cod_IBGE"
    ].isna().sum()
)

print(
    "Documentos duplicados na dimensão:",
    dim_credor_cnpj[
        "Documento_Original"
    ].duplicated().sum()
)

print(
    "Diferença financeira dos repasses:",
    diferenca_financeira_repasse
)

VALIDAÇÃO FINAL DO MODELO
------------------------------------------------------------
DIM_MUNICIPIO: 497
FACT_IMPACTO_MUNICIPIO: 68
FACT_REPASSE_MUNICIPIO: 5868
FACT_PAGAMENTOS: 38210
DIM_CREDOR_CNPJ: 8023
DIM_INDICADORES_SOCIAIS_RS: 1
------------------------------------------------------------
Impactos sem Código IBGE: 0
Repasses territorializados sem Código IBGE: 2
Documentos duplicados na dimensão: 0
Diferença financeira dos repasses: 0.0


# Exportação do resumo final

In [ ]:
arquivo_resumo = os.path.join(
    pasta_projeto,
    "RESUMO_MODELO_ANALITICO.xlsx"
)

resumo_modelo.to_excel(
    arquivo_resumo,
    index=False
)

print(
    "✅ RESUMO_MODELO_ANALITICO.xlsx "
    "atualizado com sucesso."
)

✅ RESUMO_MODELO_ANALITICO.xlsx atualizado com sucesso.


# 12. Estrutura Final do Modelo Analítico

## Objetivo

Apresentar a estrutura final das tabelas preparadas para carga no Qlik Cloud.

O modelo foi construído para permitir análises financeiras, territoriais, sociais e temporais relacionadas ao processo de reconstrução do Rio Grande do Sul após os eventos climáticos extremos.

A granularidade indica o que cada linha representa em uma tabela. Portanto, “um registro por município” não significa que a tabela possui somente uma linha, mas que cada município aparece uma única vez.

---

## Tabelas Produzidas

### DIM_MUNICIPIO

Dimensão territorial principal do modelo.

**Granularidade:**

Cada registro representa um único município do Rio Grande do Sul.

**Quantidade final:**

497 registros, correspondentes aos 497 municípios do estado.

**Chave primária:**

`Cod_IBGE`

**Aplicações:**

- Centralizar os atributos territoriais e demográficos dos municípios
- Relacionar os indicadores de impacto territorial
- Relacionar os pagamentos e repasses associados aos municípios
- Permitir análises comparativas entre população, impacto e recursos territorializados

---

### FACT_IMPACTO_MUNICIPIO

Tabela com os indicadores consolidados de impacto territorial.

Os diferentes clusters associados ao mesmo município foram agrupados para produzir uma única observação municipal.

**Granularidade:**

Cada registro representa um único município impactado.

**Quantidade final:**

68 registros, correspondentes aos 68 municípios identificados na base de impacto.

**Chave de relacionamento:**

`Cod_IBGE`

**Aplicações:**

- Analisar a população afetada
- Analisar a área territorial afetada
- Analisar a densidade populacional das áreas atingidas
- Identificar a recorrência dos eventos
- Comparar o impacto municipal com os recursos pagos ou repassados

---

### FACT_REPASSE_MUNICIPIO

Tabela de pagamentos e repasses territorializados conforme disponibilizado no Painel da Transparência da Crise Climática do Rio Grande do Sul.

A tabela contém os registros financeiros associados a municípios, regiões COREDE, órgãos, projetos, subprojetos, funções, credores e números de empenho.

**Granularidade:**

Cada registro representa uma ocorrência de pagamento ou repasse territorializado conforme apresentado na fonte oficial.

O número do empenho não representa uma chave única, pois um mesmo empenho pode aparecer em diferentes registros, parcelas, datas ou municípios.

**Quantidade final:**

5.801 registros financeiros.

**Chave técnica:**

`ID_Repasse_Municipio`

**Chave de relacionamento territorial:**

`Cod_IBGE`

**Cobertura territorial:**

- 4.073 registros com município identificado
- 1.728 registros com município não identificado
- 409 municípios informados na fonte antes da validação com o Código IBGE

**Valor financeiro total:**

R$ 3.309.604.256,47.

**Distribuição territorial do valor:**

- R\$ 668.913.323,65 — Município identificado
- R\$ 2.640.690.932,82 — Município não identificado

Os registros sem município identificado foram preservados na tabela e não foram distribuídos artificialmente entre os municípios.

**Aplicações:**

- Analisar o valor pago ou repassado por município
- Identificar os municípios com maior volume financeiro associado
- Comparar recursos territorializados e impacto observado
- Analisar os valores por Região COREDE
- Avaliar a distribuição por situação municipal
- Analisar os recursos por órgão, função, projeto, subprojeto e credor
- Medir o percentual de recursos sem identificação municipal
- Avaliar a evolução temporal dos pagamentos e repasses

**Limitação metodológica:**

O campo `Municipio_Repasse` representa o município associado ao registro pelo Painel da Transparência.

O valor não representa necessariamente uma transferência direta para a prefeitura, pois a base também inclui pagamentos a empresas, entidades, fundos municipais e outros credores por ações executadas ou associadas ao território.

Por esse motivo, o indicador será descrito como:

`Valor pago ou repassado associado ao município`

Não será descrito genericamente como:

`Valor recebido pela prefeitura`

---

### FACT_PAGAMENTOS

Tabela completa de execução financeira relacionada às ações de resposta e reconstrução.

**Granularidade:**

Cada registro representa um único lançamento financeiro presente na base de pagamentos.

Um mesmo órgão, credor, projeto ou subprojeto pode aparecer em diversos registros, pois pode estar associado a diferentes pagamentos.

**Quantidade final:**

37.912 registros financeiros.

**Chave de relacionamento cadastral:**

`Documento_Original`

**Aplicações:**

- Analisar o valor líquido executado
- Avaliar a evolução temporal dos gastos
- Identificar os principais órgãos executores
- Identificar os principais credores
- Analisar projetos, subprojetos e tipos de despesa
- Preservar os estornos, devoluções e ajustes contábeis
- Relacionar os lançamentos aos dados cadastrais dos credores

---

### DIM_CREDOR_CNPJ

Dimensão cadastral dos credores identificados na base financeira.

**Granularidade:**

Cada registro representa um documento único identificado na base de pagamentos.

**Quantidade final:**

8.023 registros válidos.

**Chave primária:**

`Documento_Original`

**Relacionamento:**

A dimensão se relaciona com a `FACT_PAGAMENTOS` através do campo `Documento_Original`, em uma relação de um credor para vários pagamentos.

**Aplicações:**

- Identificar os CNPJs localizados
- Analisar a situação cadastral dos credores
- Identificar o município e a Unidade da Federação cadastral do credor
- Analisar a atividade econômica principal
- Avaliar a concentração financeira por empresa ou entidade
- Diferenciar credores do Rio Grande do Sul e de outros estados

**Limitação metodológica:**

O campo `Municipio_Credor` representa a localização cadastral do estabelecimento que recebeu o pagamento.

Esse campo não representa necessariamente o município beneficiado nem o local onde o recurso foi aplicado.

A análise territorial principal dos recursos será realizada através da `FACT_REPASSE_MUNICIPIO`.

---

### DIM_INDICADORES_SOCIAIS_RS

Dimensão de contexto socioeconômico e de desenvolvimento humano do Rio Grande do Sul.

**Granularidade:**

O registro representa os indicadores consolidados da Unidade da Federação do Rio Grande do Sul para o ano de 2024.

**Quantidade final:**

1 registro estadual.

**Aplicações:**

- Contextualizar o nível de desenvolvimento humano do estado
- Apresentar indicadores de renda, educação, pobreza e desigualdade
- Apoiar a interpretação dos resultados territoriais e financeiros
- Fornecer contexto social para a análise da capacidade de recuperação estadual

A dimensão não possui relacionamento direto com os municípios, pois os indicadores representam o Rio Grande do Sul como um todo.

---

## Modelo Conceitual

```text
                         NÚCLEO TERRITORIAL

                         DIM_MUNICIPIO
                         │           │
                Cod_IBGE│           │Cod_IBGE
                         ▼           ▼
          FACT_IMPACTO_MUNICIPIO   FACT_REPASSE_MUNICIPIO


                         NÚCLEO FINANCEIRO

                         DIM_CREDOR_CNPJ
                                │
             Documento_Original │
                                ▼
                         FACT_PAGAMENTOS


                         CONTEXTO ESTADUAL

                  DIM_INDICADORES_SOCIAIS_RS
```

---

## Relacionamentos do Modelo

### Relacionamento Territorial de Impacto

```text
DIM_MUNICIPIO.Cod_IBGE
=
FACT_IMPACTO_MUNICIPIO.Cod_IBGE
```

**Cardinalidade esperada:**

```text
DIM_MUNICIPIO 1 : 0..1 FACT_IMPACTO_MUNICIPIO
```

A tabela de impacto contém apenas os municípios identificados na base de clusters, enquanto a dimensão contém todos os municípios do Rio Grande do Sul.

---

### Relacionamento Territorial dos Recursos

```text
DIM_MUNICIPIO.Cod_IBGE
=
FACT_REPASSE_MUNICIPIO.Cod_IBGE
```

**Cardinalidade esperada:**

```text
DIM_MUNICIPIO 1 : N FACT_REPASSE_MUNICIPIO
```

Um município pode estar associado a diversos pagamentos e repasses.

Os registros classificados como município indefinido permanecerão sem Código IBGE e, portanto, não serão associados artificialmente à dimensão municipal.

---

### Relacionamento Financeiro e Cadastral

```text
DIM_CREDOR_CNPJ.Documento_Original
=
FACT_PAGAMENTOS.Documento_Original
```

**Cardinalidade esperada:**

```text
DIM_CREDOR_CNPJ 1 : N FACT_PAGAMENTOS
```

Um documento pode estar associado a diversos lançamentos financeiros.

---

## Separação entre os Conceitos Municipais

O modelo possui dois conceitos territoriais distintos.

### Municipio_Repasse

Representa o município associado ao pagamento ou repasse pelo Painel da Transparência.

Esse campo será utilizado para:

- Distribuição territorial dos recursos
- Comparação entre impacto e valor financeiro
- Análise por Região COREDE
- Análise por situação municipal
- Construção de mapas de recursos territorializados

### Municipio_Credor

Representa o município cadastral do estabelecimento ou entidade que recebeu o pagamento.

Esse campo será utilizado para:

- Localização dos fornecedores
- Concentração territorial dos credores
- Análise de empresas do Rio Grande do Sul e de outros estados
- Análise cadastral dos recebedores

Os dois campos não serão tratados como equivalentes.

---

## Estratégia Analítica

### Análise de Impacto e Recursos

Será realizada através das tabelas:

```text
DIM_MUNICIPIO